<a href="https://colab.research.google.com/github/yangyi02/droid/blob/main/PointWorld_filter_and_flows.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## 🚀 Colab One-Time Setup

Run **all cells in this section once** per runtime. They clone the repo, install dependencies, and download model weights.

> **Requirements**: Use a **GPU runtime** (T4 or better). Go to `Runtime → Change runtime type → GPU`.

In [ ]:
import os

# Clone PointWorld repo with all submodules (switch to 'data' branch)
COLAB_ROOT = "/content/PointWorld"
if not os.path.exists(COLAB_ROOT):
    !git clone --recurse-submodules https://github.com/NVlabs/PointWorld.git {COLAB_ROOT}
    %cd {COLAB_ROOT}
    !git checkout data
    !git submodule update --init --recursive
else:
    %cd {COLAB_ROOT}
    print(f"PointWorld already cloned at {COLAB_ROOT}")

print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install all required Python packages
# Step 1: Upgrade numpy to 2.x first (required by opencv, jax, etc. in Colab)
!pip install -q "numpy>=2.0,<2.3"

# Step 2: Install numba (latest, supports numpy 2.x)
!pip install -q numba

# Step 3: Install remaining dependencies
!pip install -q \
    urdfpy \
    omegaconf \
    "timm==0.9.16" \
    einops \
    trimesh \
    h5py \
    tqdm \
    scipy \
    scikit-learn \
    scikit-image \
    transforms3d \
    transformations \
    huggingface_hub \
    safetensors \
    opencv-contrib-python \
    pytorch_kinematics \
    open3d \
    pyyaml

# Install VGGT from the submodule
!pip install -q {COLAB_ROOT}/third_party/vggt

# Verify numpy and numba versions are compatible
import numpy as np
import numba
print(f"\n✅ All Python dependencies installed.")
print(f"   numpy  : {np.__version__}")
print(f"   numba  : {numba.__version__}")

In [ ]:
# Install ZED SDK for SVO file decoding
import os

ZED_SDK_INSTALLED = False
try:
    import pyzed.sl as sl
    ZED_SDK_INSTALLED = True
    print("✅ ZED SDK (pyzed) already installed.")
except ImportError:
    print("ZED SDK not found. Installing...")
    # Download and install the ZED SDK (headless, for Colab/server environments)
    !wget -q --show-progress -O /tmp/zed_sdk.run \
        "https://download.stereolabs.com/zedsdk/4.2/cu121/ubuntu22" && \
        chmod +x /tmp/zed_sdk.run && \
        /tmp/zed_sdk.run -- silent skip_cuda skip_od_module skip_tools && \
        rm /tmp/zed_sdk.run
    # Install the Python bindings
    !pip install -q /usr/local/zed/pyzed*.whl 2>/dev/null || \
        python /usr/local/zed/get_python_api.py
    try:
        import pyzed.sl as sl
        ZED_SDK_INSTALLED = True
        print("✅ ZED SDK installed successfully.")
    except ImportError:
        print("⚠️ ZED SDK installation failed. SVO decoding will not work.")
        print("   You can try installing manually: https://www.stereolabs.com/docs/installation/linux")

In [ ]:
import os

# Download CoTracker checkpoint from GCS
COTRACKER_DIR = os.path.join(COLAB_ROOT, "checkpoints", "cotracker")
COTRACKER_CKPT = os.path.join(COTRACKER_DIR, "scaled_online.pth")

if not os.path.exists(COTRACKER_CKPT):
    os.makedirs(COTRACKER_DIR, exist_ok=True)
    !gcloud storage cp gs://dm-tapnet/checkpoints/cotracker/scaled_online.pth {COTRACKER_CKPT}
    print(f"✅ CoTracker checkpoint downloaded to {COTRACKER_CKPT}")
else:
    print(f"⏭️ CoTracker checkpoint already exists at {COTRACKER_CKPT}")

# Verify file size
if os.path.exists(COTRACKER_CKPT):
    size_mb = os.path.getsize(COTRACKER_CKPT) / (1024 * 1024)
    print(f"   File size: {size_mb:.1f} MB")

In [ ]:
# Authenticate with Google Cloud for GCS access to DROID data
from google.colab import auth
auth.authenticate_user()

# Set GCS cache directory for DROID data
import os
os.environ['POINTWORLD_CACHE_DIR'] = '/content/pointworld_cache'
os.makedirs(os.environ['POINTWORLD_CACHE_DIR'], exist_ok=True)
print(f"✅ GCS auth complete. Cache dir: {os.environ['POINTWORLD_CACHE_DIR']}")

# Quick verification that GCS is accessible
!gcloud storage ls gs://gresearch/robotics/droid_raw/1.0.1/ 2>&1 | head -5

# Filter Scenes & Compute 2D/3D Flows for PointWorld

This notebook runs the second half of the PointWorld data pipeline:

1. **Filter Scenes by Extrinsics Quality** — Reads `<uuid>_cameras.json` files (from `compute_extrinsics`) and keeps only scenes whose optimization `final_loss < threshold`, producing a filtered scene list.
2. **Compute 2D Flows** — Runs CoTracker on each scene to produce 2D point tracks across video clips, saved per-scene as `<uuid>_2d_flows.h5`.
3. **Convert 2D Flows to 3D** — Loads cached 2D flows + depth + extrinsics, applies robot/workspace masks, lifts 2D tracks to 3D, removes outliers, and writes the final `<uuid>_flows.h5` consumed by the training pipeline.

> **Prerequisites**: You must have **already run** the `PointWorld_compute_depth_and_extrinsics.ipynb` notebook to produce `depth/` and `cameras/` outputs under `$DROID_ROOT`.

> **How to use**: Run all cells top-to-bottom. The Colab Setup section above handles all dependencies automatically.

---
## Step 0 — Environment Setup

Set up `sys.path` so the PointWorld modules are importable, and create the required directory structure.

In [ ]:
import sys, os

# Point to the cloned PointWorld repo
REPO_ROOT = "/content/PointWorld"
REAL_DIR = os.path.join(REPO_ROOT, "real")

# Ensure repo root and real/ are importable
for p in [REPO_ROOT, REAL_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("REPO_ROOT:", REPO_ROOT)
print("sys.path entries:", sys.path[:5])

---
## Step 1 — Define: `gcs_utils` (GCS helpers)

In [ ]:
# ============================================================
# gcs_utils — Google Cloud Storage helpers
# ============================================================
import os, re, tempfile, subprocess

def is_gcs_path(path):
    """Check if the path is a Google Cloud Storage path."""
    return path.startswith('gs://')

def parse_gcs_path(gcs_path):
    """Parse a GCS path into bucket and blob names."""
    match = re.match(r'gs://([^/]+)/(.*)', gcs_path)
    if not match:
        raise ValueError(f"Invalid GCS path: {gcs_path}")
    return match.group(1), match.group(2)

def download_from_gcs(gcs_path, local_path):
    """Download a file from GCS to a local path using gsutil."""
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    cmd = ["gsutil", "cp", gcs_path, local_path]
    try:
        subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"GCS download failed for {gcs_path}: {e.stderr.strip()}") from e
    return local_path

def get_local_path(path, temp_dir=None):
    """If GCS path, download to local cache or temp dir and return local path."""
    if not is_gcs_path(path):
        return path
    cache_root = os.environ.get('POINTWORLD_CACHE_DIR')
    if cache_root:
        bucket, blob = parse_gcs_path(path)
        local_path = os.path.join(cache_root, "droid", bucket, blob)
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        if os.path.exists(local_path):
            return local_path
        download_from_gcs(path, local_path)
        return local_path
    else:
        if temp_dir is None:
            temp_dir = tempfile.mkdtemp()
        local_path = os.path.join(temp_dir, os.path.basename(path))
        download_from_gcs(path, local_path)
        return local_path

def enforce_gcs_cache_policy(scene_paths, stage_name, require_cache=False, allow_streaming=False):
    """Validate cache policy for GCS-backed scene paths."""
    if isinstance(scene_paths, str):
        scene_paths = [scene_paths]
    gcs_paths = [p for p in scene_paths if is_gcs_path(p)]
    if not gcs_paths:
        return False
    cache_root = os.environ.get("POINTWORLD_CACHE_DIR", "").strip()
    if cache_root:
        print(f"[{stage_name}] cache enabled: POINTWORLD_CACHE_DIR={cache_root}")
        return True
    message = (
        f"[{stage_name}] Detected {len(gcs_paths)} GCS scene path(s) but POINTWORLD_CACHE_DIR is not set. "
        "Without persistent caching, repeated stages may re-download the same objects."
    )
    setup_hint = "Set: export POINTWORLD_CACHE_DIR=/path/to/fast_disk/pointworld_cache"
    if require_cache and not allow_streaming:
        raise RuntimeError(f"{message} {setup_hint}")
    print(f"WARNING: {message} Continuing because allow_streaming={allow_streaming}.")
    return True

def list_gcs_files(gcs_path, pattern=None):
    """List files in a GCS directory that match a pattern using gsutil."""
    if not gcs_path.endswith('/'):
        gcs_path += '/'
    search_path = os.path.join(gcs_path, pattern) if pattern else gcs_path + '*'
    cmd = ["gsutil", "ls", search_path]
    try:
        result = subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        return [line.strip() for line in result.stdout.split('\n') if line.strip()]
    except subprocess.CalledProcessError as e:
        if "No URLs matched" in e.stderr:
            return []
        raise RuntimeError(f"Error listing files in {gcs_path}: {e.stderr}")

print("✅ gcs_utils defined")

---
## Step 2 — Define: `real_utils` (time helpers & numba kernels)

In [ ]:
# ============================================================
# real_utils — time helpers and numba projection kernels
# ============================================================
import numpy as np
from datetime import datetime

try:
    from numba import njit
    _NUMBA_AVAILABLE = True
except (ImportError, ValueError) as _numba_err:
    print(f"⚠️ numba import failed ({_numba_err}); using pure-Python fallback for projection kernel.")
    _NUMBA_AVAILABLE = False
    def njit(*args, **kwargs):
        """No-op decorator when numba is unavailable."""
        def decorator(fn): return fn
        return decorator if args and callable(args[0]) else decorator

@njit(cache=True, fastmath=True, nogil=True)
def project_points_to_image(points_3d, transform_matrix, extrinsic, intrinsic, image_width, image_height):
    """Fused kernel to project 3D points to 2D image coordinates."""
    n_points = points_3d.shape[0]
    projected_points = np.empty((n_points, 2), dtype=np.float32)
    valid_count = 0
    for i in range(n_points):
        local_point = np.array([points_3d[i, 0], points_3d[i, 1], points_3d[i, 2], 1.0], dtype=np.float32)
        world_point = np.zeros(4, dtype=np.float32)
        for j in range(4):
            world_point[j] = (transform_matrix[j, 0] * local_point[0] +
                              transform_matrix[j, 1] * local_point[1] +
                              transform_matrix[j, 2] * local_point[2] +
                              transform_matrix[j, 3] * local_point[3])
        cam_point = np.zeros(4, dtype=np.float32)
        for j in range(4):
            cam_point[j] = (extrinsic[j, 0] * world_point[0] +
                            extrinsic[j, 1] * world_point[1] +
                            extrinsic[j, 2] * world_point[2] +
                            extrinsic[j, 3] * world_point[3])
        if cam_point[2] <= 0:
            continue
        img_x = (intrinsic[0, 0] * cam_point[0] + intrinsic[0, 2] * cam_point[2]) / cam_point[2]
        img_y = (intrinsic[1, 1] * cam_point[1] + intrinsic[1, 2] * cam_point[2]) / cam_point[2]
        if img_x >= 0 and img_x < image_width and img_y >= 0 and img_y < image_height:
            projected_points[valid_count, 0] = img_x
            projected_points[valid_count, 1] = img_y
            valid_count += 1
    return projected_points[:valid_count].copy()

@njit(cache=True, fastmath=True, nogil=True)
def generate_and_project_workspace_boundary(bounds_min, bounds_max, face_density, extrinsic, intrinsic, image_width, image_height):
    """Generate 3D workspace boundary and project to 2D."""
    # Generate boundary points on the 6 faces of the AABB
    boundary_pts_list = []
    xmin, ymin, zmin = bounds_min[0], bounds_min[1], bounds_min[2]
    xmax, ymax, zmax = bounds_max[0], bounds_max[1], bounds_max[2]
    # Number of points per edge
    n = face_density
    projected = np.empty((6 * n * n, 2), dtype=np.float32)
    count = 0
    for face in range(6):
        for i in range(n):
            for j in range(n):
                u = i / max(1, n - 1)
                v = j / max(1, n - 1)
                if face == 0:  # x = xmin
                    pt = np.array([xmin, ymin + u*(ymax-ymin), zmin + v*(zmax-zmin), 1.0], dtype=np.float32)
                elif face == 1:  # x = xmax
                    pt = np.array([xmax, ymin + u*(ymax-ymin), zmin + v*(zmax-zmin), 1.0], dtype=np.float32)
                elif face == 2:  # y = ymin
                    pt = np.array([xmin + u*(xmax-xmin), ymin, zmin + v*(zmax-zmin), 1.0], dtype=np.float32)
                elif face == 3:  # y = ymax
                    pt = np.array([xmin + u*(xmax-xmin), ymax, zmin + v*(zmax-zmin), 1.0], dtype=np.float32)
                elif face == 4:  # z = zmin
                    pt = np.array([xmin + u*(xmax-xmin), ymin + v*(ymax-ymin), zmin, 1.0], dtype=np.float32)
                else:  # z = zmax
                    pt = np.array([xmin + u*(xmax-xmin), ymin + v*(ymax-ymin), zmax, 1.0], dtype=np.float32)
                # Project: cam_point = extrinsic @ pt
                cam_point = np.zeros(4, dtype=np.float32)
                for k in range(4):
                    cam_point[k] = (extrinsic[k, 0] * pt[0] +
                                    extrinsic[k, 1] * pt[1] +
                                    extrinsic[k, 2] * pt[2] +
                                    extrinsic[k, 3] * pt[3])
                if cam_point[2] <= 0:
                    continue
                img_x = (intrinsic[0, 0] * cam_point[0] + intrinsic[0, 2] * cam_point[2]) / cam_point[2]
                img_y = (intrinsic[1, 1] * cam_point[1] + intrinsic[1, 2] * cam_point[2]) / cam_point[2]
                if img_x >= 0 and img_x < image_width and img_y >= 0 and img_y < image_height:
                    projected[count, 0] = img_x
                    projected[count, 1] = img_y
                    count += 1
    return projected[:count].copy()

def get_mesh_name(mesh, idx):
    try:
        return f'{mesh.source.file_name.lower()}_{idx}'
    except AttributeError:
        return f'{mesh.metadata.get("name", mesh.metadata.get("file_name", f"unknown")).lower()}_{idx}'

def get_time_str():
    """Return current time in the format: YYYY-MM-DD HH:MM:SS"""
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')

print(f"✅ real_utils defined (numba available: {_NUMBA_AVAILABLE})")

---
## Step 3 — Define: `transform_utils` (rotation / pose math)

In [ ]:
# ============================================================
# transform_utils — matrix and vector transformation utilities
# ============================================================
import math
import numpy as np
from scipy.spatial.transform import Rotation as R

try:
    from numba import njit
    _TU_NUMBA_AVAILABLE = True
except (ImportError, ValueError) as _numba_err:
    _TU_NUMBA_AVAILABLE = False
    def njit(*args, **kwargs):
        def decorator(fn): return fn
        return decorator if args and callable(args[0]) else decorator

PI = np.pi
EPS = np.finfo(float).eps * 4.0

@njit(cache=True, fastmath=True)
def _mat2quat_single(Rmat):
    t = Rmat[0,0] + Rmat[1,1] + Rmat[2,2]
    if t > 0.0:
        s = 0.5 / np.sqrt(t + 1.0)
        w = 0.25 / s
        x = (Rmat[2,1] - Rmat[1,2]) * s
        y = (Rmat[0,2] - Rmat[2,0]) * s
        z = (Rmat[1,0] - Rmat[0,1]) * s
    else:
        if Rmat[0,0] > Rmat[1,1] and Rmat[0,0] > Rmat[2,2]:
            s = 2.0 * np.sqrt(1.0 + Rmat[0,0] - Rmat[1,1] - Rmat[2,2])
            w = (Rmat[2,1] - Rmat[1,2]) / s; x = 0.25 * s
            y = (Rmat[0,1] + Rmat[1,0]) / s; z = (Rmat[0,2] + Rmat[2,0]) / s
        elif Rmat[1,1] > Rmat[2,2]:
            s = 2.0 * np.sqrt(1.0 + Rmat[1,1] - Rmat[0,0] - Rmat[2,2])
            w = (Rmat[0,2] - Rmat[2,0]) / s; x = (Rmat[0,1] + Rmat[1,0]) / s
            y = 0.25 * s; z = (Rmat[1,2] + Rmat[2,1]) / s
        else:
            s = 2.0 * np.sqrt(1.0 + Rmat[2,2] - Rmat[0,0] - Rmat[1,1])
            w = (Rmat[1,0] - Rmat[0,1]) / s; x = (Rmat[0,2] + Rmat[2,0]) / s
            y = (Rmat[1,2] + Rmat[2,1]) / s; z = 0.25 * s
    return np.array((x, y, z, w), dtype=Rmat.dtype)

@njit(cache=True, fastmath=True)
def _quat2mat_single(q):
    x, y, z, w = q
    xx, yy, zz = x*x, y*y, z*z
    xy, xz, yz = x*y, x*z, y*z
    wx, wy, wz = w*x, w*y, w*z
    M = np.empty((3, 3), dtype=q.dtype)
    M[0,0]=1.0-2.0*(yy+zz); M[0,1]=2.0*(xy-wz); M[0,2]=2.0*(xz+wy)
    M[1,0]=2.0*(xy+wz); M[1,1]=1.0-2.0*(xx+zz); M[1,2]=2.0*(yz-wx)
    M[2,0]=2.0*(xz-wy); M[2,1]=2.0*(yz+wx); M[2,2]=1.0-2.0*(xx+yy)
    return M

@njit(cache=True, fastmath=True)
def _mat2quat_kernel(poses_mat, out):
    N = poses_mat.shape[0]
    for i in range(N):
        out[i, :3] = poses_mat[i, :3, 3]
        out[i, 3:] = _mat2quat_single(poses_mat[i, :3, :3])

@njit(cache=True, fastmath=True)
def _quat2mat_kernel(poses_quat, out):
    N = poses_quat.shape[0]
    for i in range(N):
        out[i, :3, 3] = poses_quat[i, :3]
        Rm = _quat2mat_single(poses_quat[i, 3:])
        out[i, 0, :3] = Rm[0]; out[i, 1, :3] = Rm[1]; out[i, 2, :3] = Rm[2]

def convert_pose_mat2quat(poses_mat):
    was_single = poses_mat.ndim == 2
    if was_single: poses_mat = poses_mat[None]
    out = np.empty((poses_mat.shape[0], 7), dtype=poses_mat.dtype)
    _mat2quat_kernel(poses_mat, out)
    return out[0] if was_single else out

def convert_pose_quat2mat(poses_quat):
    was_single = poses_quat.ndim == 1
    if was_single: poses_quat = poses_quat[None]
    out = np.empty((poses_quat.shape[0], 4, 4), dtype=poses_quat.dtype)
    out[:, 3, :] = np.array((0., 0., 0., 1.), dtype=poses_quat.dtype)
    _quat2mat_kernel(poses_quat, out)
    return out[0] if was_single else out

def mat2quat(rmat): return R.from_matrix(rmat).as_quat()
def quat2mat(quaternion): return R.from_quat(quaternion).as_matrix()
def euler2mat(euler): return R.from_euler("xyz", np.asarray(euler, dtype=np.float64)).as_matrix()
def mat2euler(rmat): return R.from_matrix(np.array(rmat)[:3, :3]).as_euler("xyz")
def euler2quat(euler): return R.from_euler("xyz", euler).as_quat()
def quat2euler(quat): return R.from_quat(quat).as_euler("xyz")
def quat2axisangle(quat): return R.from_quat(quat).as_rotvec()
def axisangle2quat(vec): return R.from_rotvec(vec).as_quat()

def mat2pose(hmat):
    return hmat[:3, 3], mat2quat(hmat[:3, :3])

def pose2mat(pose):
    homo = np.zeros((4, 4), dtype=pose[0].dtype)
    homo[:3, :3] = quat2mat(pose[1]); homo[:3, 3] = np.array(pose[0]); homo[3, 3] = 1.0
    return homo

def pose_inv(pose_mat):
    out = np.zeros((4, 4))
    out[:3, :3] = pose_mat[:3, :3].T
    out[:3, 3] = -out[:3, :3].dot(pose_mat[:3, 3])
    out[3, 3] = 1.0
    return out

def quat_multiply(quaternion1, quaternion0):
    x0,y0,z0,w0 = quaternion0; x1,y1,z1,w1 = quaternion1
    return np.array((
        x1*w0+y1*z0-z1*y0+w1*x0, -x1*z0+y1*w0+z1*x0+w1*y0,
        x1*y0-y1*x0+z1*w0+w1*z0, -x1*x0-y1*y0-z1*z0+w1*w0,
    ), dtype=quaternion0.dtype)

def quat_conjugate(q): return np.array((-q[0], -q[1], -q[2], q[3]), dtype=q.dtype)
def quat_inverse(q): return quat_conjugate(q) / np.dot(q, q)
def quat_distance(q1, q0): return quat_multiply(q1, quat_inverse(q0))

def convert_pose_euler2mat(poses_euler):
    was_single = poses_euler.ndim == 1
    if was_single: poses_euler = poses_euler[None]
    N = poses_euler.shape[0]
    out = np.eye(4)[None].repeat(N, axis=0)
    out[:, :3, 3] = poses_euler[:, :3]
    for i in range(N):
        out[i, :3, :3] = euler2mat(poses_euler[i, 3:])
    return out[0] if was_single else out

def convert_pose_euler2quat(poses_euler):
    mats = convert_pose_euler2mat(poses_euler)
    return convert_pose_mat2quat(mats)

def get_pose_error(pose1, pose0):
    """Compute [dx, dy, dz, drx, dry, drz] error between two 4x4 poses."""
    T_delta = np.linalg.inv(pose0) @ pose1
    trans_err = T_delta[:3, 3]
    rot_err = R.from_matrix(T_delta[:3, :3]).as_rotvec()
    return np.concatenate([trans_err, rot_err])

print(f"✅ transform_utils defined (numba available: {_TU_NUMBA_AVAILABLE})")

---
## Step 4 — Define: `droid_utils` (DROID dataset loading)

In [ ]:
# ============================================================
# droid_utils — DROID dataset loading utilities
# ============================================================
import os, json, glob, tempfile, shutil
import numpy as np
import cv2
from tqdm import tqdm
import h5py

try:
    import pyzed.sl as sl
except ImportError as exc:
    sl = None
    _PYZED_IMPORT_ERROR = exc
else:
    _PYZED_IMPORT_ERROR = None

def _require_pyzed():
    if sl is None:
        raise ImportError("pyzed is required for ZED SVO processing") from _PYZED_IMPORT_ERROR

def _binary_search_latest_range(arr, left, right, target):
    if arr[right] <= target or right == left:
        return arr[right]
    mid = ((left + right) >> 1) + 1
    if arr[mid] <= target:
        return _binary_search_latest_range(arr, mid, right, target)
    return _binary_search_latest_range(arr, left, mid - 1, target)

def _binary_search_latest(arr, target):
    if len(arr) <= 0:
        raise ValueError("input array should contain at least one element")
    return _binary_search_latest_range(arr, 0, len(arr) - 1, target)

def _binary_search_closest(arr, target):
    if target in arr:
        return target
    prev_idx = arr.index(_binary_search_latest(arr, target))
    if prev_idx == len(arr) - 1:
        return arr[prev_idx]
    prev_val = arr[prev_idx]; next_val = arr[prev_idx + 1]
    return prev_val if abs(prev_val - target) < abs(next_val - target) else next_val

def _resolve_svo_path(scene_path, svo_path):
    if not svo_path.startswith('/') and not is_gcs_path(svo_path):
        return os.path.join(scene_path, *svo_path.split('/')[-3:])
    return svo_path

def _init_camera_entries(metadata, include_wrist_cam):
    camera_names = ['wrist', 'ext1', 'ext2'] if include_wrist_cam else ['ext1', 'ext2']
    data_dict = {}; camera_specs = []
    for camera_name in camera_names:
        serial_key = f'{camera_name}_cam_serial'
        if serial_key not in metadata:
            raise KeyError(f"Camera {camera_name} not found in metadata")
        camera_serial = metadata[serial_key]
        data_dict[camera_serial] = {}
        if camera_name == 'wrist':
            data_dict[camera_serial]['extrinsic'] = np.eye(4)
        else:
            extrinsic_key = f'{camera_name}_cam_extrinsics'
            if extrinsic_key not in metadata:
                raise KeyError(f"Extrinsics for camera {camera_name} not found in metadata")
            extrinsic_6 = np.array(metadata[extrinsic_key])
            extrinsic_4x4 = convert_pose_euler2mat(extrinsic_6[None])[0]
            extrinsic_4x4 = np.linalg.inv(extrinsic_4x4)
            data_dict[camera_serial]['extrinsic'] = extrinsic_4x4
        camera_specs.append((camera_name, camera_serial))
    return data_dict, camera_specs

def _open_svo_camera(local_svo_path, include_depth):
    _require_pyzed()
    init_params = sl.InitParameters()
    init_params.set_from_svo_file(local_svo_path)
    init_params.svo_real_time_mode = False
    init_params.depth_mode = sl.DEPTH_MODE.ULTRA if include_depth else sl.DEPTH_MODE.NONE
    init_params.coordinate_units = sl.UNIT.METER
    zed = sl.Camera()
    err = zed.open(init_params)
    if err != sl.ERROR_CODE.SUCCESS:
        raise ValueError(f"Error opening SVO file {local_svo_path}: {err}")
    return zed, zed.get_camera_information()

def _extract_intrinsics_and_baseline(calibration_params, downscale_ratio):
    fx = calibration_params.left_cam.fx; fy = calibration_params.left_cam.fy
    cx = calibration_params.left_cam.cx; cy = calibration_params.left_cam.cy
    intrinsic = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])
    baseline = calibration_params.stereo_transform.get_translation().get()[0]
    if downscale_ratio != 1.0:
        intrinsic[0, 0] *= downscale_ratio; intrinsic[1, 1] *= downscale_ratio
        intrinsic[0, 2] *= downscale_ratio; intrinsic[1, 2] *= downscale_ratio
    return intrinsic, baseline

def _extract_svo_frames(zed, camera_name, include_stereo, include_depth, downscale_ratio, max_frames):
    rgb_frames = []; right_frames = [] if include_stereo else None
    depth_frames = [] if include_depth else None
    left_image = sl.Mat()
    right_image = sl.Mat() if include_stereo else None
    depth_image = sl.Mat() if include_depth else None
    runtime_params = sl.RuntimeParameters()
    nb_frames = zed.get_svo_number_of_frames()
    if max_frames > 0: nb_frames = min(nb_frames, max_frames)
    timestamps = []; frame_count = 0
    with tqdm(total=nb_frames, desc=f"Processing {camera_name} camera") as pbar:
        while frame_count < nb_frames:
            err = zed.grab(runtime_params)
            if err == sl.ERROR_CODE.SUCCESS:
                zed.retrieve_image(left_image, sl.VIEW.LEFT)
                rgb = left_image.get_data().copy()
                if rgb.shape[2] == 4: rgb = cv2.cvtColor(rgb, cv2.COLOR_BGRA2RGB)
                if downscale_ratio != 1.0: rgb = cv2.resize(rgb, None, fx=downscale_ratio, fy=downscale_ratio, interpolation=cv2.INTER_LINEAR)
                rgb_frames.append(rgb)
                ts = zed.get_timestamp(sl.TIME_REFERENCE.IMAGE).get_milliseconds()
                timestamps.append(ts)
                if include_stereo:
                    zed.retrieve_image(right_image, sl.VIEW.RIGHT)
                    right = right_image.get_data().copy()
                    if right.shape[2] == 4: right = cv2.cvtColor(right, cv2.COLOR_BGRA2RGB)
                    if downscale_ratio != 1.0: right = cv2.resize(right, None, fx=downscale_ratio, fy=downscale_ratio, interpolation=cv2.INTER_LINEAR)
                    right_frames.append(right)
                if include_depth:
                    zed.retrieve_measure(depth_image, sl.MEASURE.DEPTH)
                    depth = depth_image.get_data().copy()
                    if downscale_ratio != 1.0: depth = cv2.resize(depth, None, fx=downscale_ratio, fy=downscale_ratio, interpolation=cv2.INTER_NEAREST)
                    depth_frames.append(depth)
                frame_count += 1; pbar.update(1)
            elif err == sl.ERROR_CODE.END_OF_SVOFILE_REACHED:
                print(f"End of SVO file reached for camera {camera_name}"); break
            else:
                raise ValueError(f"Error grabbing frame from SVO for camera {camera_name}: {err}")
    if not rgb_frames: raise ValueError(f"No frames extracted for camera {camera_name}")
    payload = {'rgb': np.stack(rgb_frames, axis=0), 'timestamps': np.array(timestamps)}
    if include_stereo: payload['right_frames'] = np.stack(right_frames, axis=0)
    if include_depth:
        payload['depth'] = np.stack(depth_frames, axis=0)
        payload['depth'] = np.nan_to_num(payload['depth'], nan=0.0, posinf=0.0, neginf=0.0, copy=False)
    return payload

def filter_by_timestamps(data, canonical_timestamps, scene_path, is_proprio=False, verbose=False):
    filtered_data = {}
    if is_proprio:
        temp_dir = None
        try:
            if is_gcs_path(scene_path): temp_dir = tempfile.mkdtemp()
            trajectory_path = os.path.join(scene_path, "trajectory.h5")
            local_trajectory_path = get_local_path(trajectory_path, temp_dir)
            with h5py.File(local_trajectory_path, 'r') as f:
                proprio_timestamps = np.array(f['observation']['timestamp']['robot_state']['read_end'])
        finally:
            if temp_dir and os.path.exists(temp_dir): shutil.rmtree(temp_dir)
        proprio_timestamps_list = proprio_timestamps.tolist()
        for modality in data:
            if modality in ['pre_sampled_points']:
                filtered_data[modality] = data[modality]
            else:
                assert len(data[modality]) == len(proprio_timestamps_list)
                aligned_data = []
                for target_ts in canonical_timestamps:
                    closest_ts = _binary_search_closest(proprio_timestamps_list, target_ts)
                    idx = proprio_timestamps_list.index(closest_ts)
                    aligned_data.append(data[modality][idx])
                filtered_data[modality] = np.array(aligned_data)
    else:
        nontemporal_keys = ['intrinsic', 'extrinsic', 'baseline']
        for camera_serial in data:
            filtered_data[camera_serial] = {}
            for key in nontemporal_keys:
                if key in data[camera_serial]:
                    filtered_data[camera_serial][key] = data[camera_serial][key]
            camera_timestamps_list = data[camera_serial]['timestamps'].tolist()
            for key in data[camera_serial]:
                if key in nontemporal_keys: continue
                assert len(data[camera_serial][key]) == len(camera_timestamps_list)
                aligned_data = []
                for target_ts in canonical_timestamps:
                    closest_ts = _binary_search_closest(camera_timestamps_list, target_ts)
                    idx = camera_timestamps_list.index(closest_ts)
                    aligned_data.append(data[camera_serial][key][idx])
                filtered_data[camera_serial][key] = np.array(aligned_data)
    return filtered_data

def get_uuid(scene_path):
    if is_gcs_path(scene_path):
        metadata_files = list_gcs_files(scene_path, "metadata_*.json")
        if not metadata_files: raise FileNotFoundError(f"No metadata files found in {scene_path}")
        return os.path.basename(metadata_files[0])[9:-5]
    metadata_files = glob.glob(os.path.join(scene_path, "metadata_*.json"))
    if not metadata_files: raise FileNotFoundError(f"No metadata files found in {scene_path}")
    return os.path.basename(metadata_files[0])[9:-5]

def get_metadata(scene_path):
    temp_dir = None
    try:
        if is_gcs_path(scene_path):
            temp_dir = tempfile.mkdtemp()
            metadata_files = list_gcs_files(scene_path, "metadata_*.json")
            if not metadata_files: raise FileNotFoundError(f"No metadata files found in {scene_path}")
            local_metadata_path = get_local_path(metadata_files[0], temp_dir)
            with open(local_metadata_path, 'r') as f: return json.load(f)
        else:
            metadata_files = glob.glob(os.path.join(scene_path, "metadata_*.json"))
            if not metadata_files: raise FileNotFoundError(f"No metadata files found in {scene_path}")
            with open(metadata_files[0], 'r') as f: return json.load(f)
    finally:
        if temp_dir and os.path.exists(temp_dir): shutil.rmtree(temp_dir)

def gather_data_dict(scene_path, downscale_ratio=1.0, include_stereo=False, include_depth=True, include_wrist_cam=False, max_frames=-1):
    _require_pyzed()
    data_dict = {}; temp_dir = None
    try:
        if is_gcs_path(scene_path): temp_dir = tempfile.mkdtemp()
        metadata = get_metadata(scene_path)
        data_dict, camera_specs = _init_camera_entries(metadata, include_wrist_cam)
        for camera_name, camera_serial in camera_specs:
            if f'{camera_name}_svo_path' not in metadata:
                raise KeyError(f"SVO path for camera {camera_name} not found in metadata")
            svo_path = metadata[f'{camera_name}_svo_path']
            resolved_svo_path = _resolve_svo_path(scene_path, svo_path)
            local_svo_path = get_local_path(resolved_svo_path, temp_dir)
            print(f"Processing SVO file for camera {camera_name}: {local_svo_path}")
            zed, camera_info = _open_svo_camera(local_svo_path, include_depth)
            try:
                calibration_params = camera_info.camera_configuration.calibration_parameters
                intrinsic, baseline = _extract_intrinsics_and_baseline(calibration_params, downscale_ratio)
                data_dict[camera_serial]['intrinsic'] = intrinsic
                if include_stereo: data_dict[camera_serial]['baseline'] = baseline
                frames_payload = _extract_svo_frames(zed, camera_name, include_stereo=include_stereo, include_depth=include_depth, downscale_ratio=downscale_ratio, max_frames=max_frames)
                data_dict[camera_serial].update(frames_payload)
            finally:
                zed.close()
            print(f"Extracted {data_dict[camera_serial]['rgb'].shape[0]} frames for camera {camera_serial}")
    finally:
        if temp_dir and os.path.exists(temp_dir): shutil.rmtree(temp_dir)
    return data_dict

def gather_trajectory(scene_path, max_frames=-1):
    temp_dir = None
    try:
        if is_gcs_path(scene_path): temp_dir = tempfile.mkdtemp()
        trajectory_path = os.path.join(scene_path, "trajectory.h5")
        local_trajectory_path = get_local_path(trajectory_path, temp_dir)
        proprio_dict = {}
        with h5py.File(local_trajectory_path, 'r') as f:
            joint_positions = np.array(f['observation']['robot_state']['joint_positions'])
            joint_velocities = np.array(f['observation']['robot_state']['joint_velocities'])
            joint_torques = np.array(f['observation']['robot_state']['joint_torques_computed'])
            gripper_positions = np.array(f['observation']['robot_state']['gripper_position'])
            gripper_pose_6 = np.array(f['observation']['robot_state']['cartesian_position'])
            gripper_pose_7 = convert_pose_euler2quat(gripper_pose_6)
            proprio_timestamps = np.array(f['observation']['timestamp']['robot_state']['read_end'])
            if max_frames > 0:
                joint_positions = joint_positions[:max_frames]
                joint_velocities = joint_velocities[:max_frames]
                joint_torques = joint_torques[:max_frames]
                gripper_positions = gripper_positions[:max_frames]
                gripper_pose_7 = gripper_pose_7[:max_frames]
                proprio_timestamps = proprio_timestamps[:max_frames]
            proprio_dict['joint_positions'] = joint_positions
            proprio_dict['joint_velocities'] = joint_velocities
            proprio_dict['joint_torques'] = joint_torques
            proprio_dict['gripper_positions'] = gripper_positions * 0.725
            proprio_dict['gripper_pose'] = gripper_pose_7
            proprio_dict['timestamps'] = proprio_timestamps
    finally:
        if temp_dir and os.path.exists(temp_dir): shutil.rmtree(temp_dir)
    return proprio_dict

def get_robot_serial(scene_path): return get_metadata(scene_path)['robot_serial']

print("✅ droid_utils defined")

---
## Step 5 — Define: `h5_io` (shared H5 read/write helpers)

In [ ]:
# ============================================================
# h5_io — H5 I/O utilities for RGB/depth storage
# ============================================================
import h5py
import numpy as np
import cv2

def save_rgb_as_jpeg_in_h5(group, dataset_name, rgb_image, jpeg_quality=95):
    """Save RGB image as JPEG binary data in HDF5 dataset."""
    assert rgb_image.dtype == np.uint8
    assert rgb_image.ndim == 3 and rgb_image.shape[2] == 3
    success, encoded_img = cv2.imencode(".jpg", rgb_image[..., ::-1], [cv2.IMWRITE_JPEG_QUALITY, jpeg_quality])
    assert success, "Failed to encode image as JPEG"
    jpeg_data = np.frombuffer(encoded_img.tobytes(), dtype=np.uint8)
    dt = h5py.special_dtype(vlen=np.dtype("uint8"))
    dset = group.create_dataset(dataset_name, (1,), dtype=dt)
    dset[0] = jpeg_data
    dset.attrs["write_complete"] = True
    dset.attrs["format"] = "jpeg"
    dset.attrs["quality"] = jpeg_quality
    dset.attrs["original_shape"] = rgb_image.shape

def load_rgb_from_jpeg_in_h5(dataset):
    """Load RGB image from JPEG binary data stored in HDF5 dataset."""
    jpeg_data = dataset[0]
    decoded_img = cv2.imdecode(jpeg_data, cv2.IMREAD_COLOR)
    if decoded_img is None:
        raise RuntimeError("Failed to decode JPEG data")
    return decoded_img[..., ::-1]

def save_depth_as_uint16_mm(group, dataset_name, depth_image_meters):
    """Save depth image as uint16 in millimeters."""
    assert depth_image_meters.ndim == 2
    depth_mm = np.clip(depth_image_meters * 1000.0, 0, 65535)
    depth_uint16 = depth_mm.astype(np.uint16)
    dset = group.create_dataset(dataset_name, data=depth_uint16, dtype=np.uint16, compression="gzip", compression_opts=4)
    dset.attrs["write_complete"] = True
    dset.attrs["format"] = "uint16_mm"
    dset.attrs["scale"] = "millimeters"

def load_depth_from_uint16_mm(dataset):
    """Load depth image from uint16 mm and convert to meters."""
    return dataset[()].astype(np.float32) / 1000.0

print("✅ h5_io defined")

---
## Step 6 — Define: `workspace` constants

In [ ]:
# ============================================================
# workspace — DROID workspace bounds (meters)
# ============================================================
import numpy as np

WORKSPACE_BOUNDS_MIN = np.array([0.00, -0.40, -0.30], dtype=np.float32)
WORKSPACE_BOUNDS_MAX = np.array([0.70, 0.40, 1.20], dtype=np.float32)

_rng = WORKSPACE_BOUNDS_MAX - WORKSPACE_BOUNDS_MIN
WORKSPACE_BOUNDS_MIN_RELAXED = WORKSPACE_BOUNDS_MIN - 0.5 * _rng
WORKSPACE_BOUNDS_MAX_RELAXED = WORKSPACE_BOUNDS_MAX + 0.5 * _rng

print(f"✅ workspace bounds defined")
print(f"   Tight:   min={WORKSPACE_BOUNDS_MIN}, max={WORKSPACE_BOUNDS_MAX}")
print(f"   Relaxed: min={WORKSPACE_BOUNDS_MIN_RELAXED}, max={WORKSPACE_BOUNDS_MAX_RELAXED}")

---
---
# Stage A — Filter Scenes by Extrinsics Quality

This stage reads the list of scene paths and evaluates the `<uuid>_cameras.json` output from `compute_extrinsics`. Scenes are **kept** only if:
- No `error_info` key is present (i.e. extrinsics optimization succeeded)
- `optimization_summary.final_loss < max_final_loss` threshold

The output is a filtered text file that is reused by both Stage B (2D flows) and Stage C (3D flows).

**Source:** `real/filter_paths_by_extrinsics_quality.py`

### Stage A — Configuration

In [ ]:
# ============================================================
# Stage A Configuration — edit these values
# ============================================================
import os

# --- REQUIRED: Set DROID_ROOT to the output directory from depth/extrinsics ---
# This should contain cameras/ and depth/ subdirectories.
DROID_ROOT = "/content/pointworld_cache/droid_output"  # ← EDIT THIS

# Input scene list (same one used for depth/extrinsics)
INPUT_SCENE_LIST = os.path.join(COLAB_ROOT, "real", "droid_paths.txt")
# For quick testing, use the debug list:
# INPUT_SCENE_LIST = os.path.join(COLAB_ROOT, "real", "droid_paths_debug.txt")

# GCS prefix for relative scene paths (empty string if paths are already absolute)
SCENE_PATH_PREFIX = "gs://gresearch/robotics/droid_raw/1.0.1/"  # ← EDIT if needed

# Output filtered list
FILTERED_OUTPUT_PATH = os.path.join(DROID_ROOT, "droid_paths_final_loss_lt_0.10.txt")

# Quality threshold
MAX_FINAL_LOSS = 0.10

# Allow streaming from GCS (vs. requiring local cache)
ALLOW_GCS_STREAMING = True

print(f"DROID_ROOT:     {DROID_ROOT}")
print(f"Input list:     {INPUT_SCENE_LIST}")
print(f"Output list:    {FILTERED_OUTPUT_PATH}")
print(f"Max final loss: {MAX_FINAL_LOSS}")

In [ ]:
# ============================================================
# Stage A — Filter scene paths by extrinsics quality
# ============================================================
import json

def filter_paths_by_extrinsics_quality(
    input_path,
    output_path,
    output_dir,
    max_final_loss=0.10,
    scene_path_prefix="",
    allow_gcs_streaming=False,
):
    """
    Read scene list, check each scene's cameras JSON, and write a filtered list.

    Each scene is kept only if:
      - cameras/<uuid>_cameras.json exists
      - No 'error_info' key (extrinsics optimization succeeded)
      - optimization_summary.final_loss < max_final_loss
    """
    # Read input scene paths
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Input scene list not found: {input_path}")
    with open(input_path, "r") as f:
        scene_paths = [line.strip() for line in f if line.strip()]
    if len(scene_paths) == 0:
        raise ValueError(f"Input scene list is empty: {input_path}")

    # Prepend GCS prefix if paths are relative
    if scene_path_prefix:
        scene_paths = [
            (scene_path_prefix + p) if not p.startswith('gs://') and not p.startswith('/') else p
            for p in scene_paths
        ]

    enforce_gcs_cache_policy(
        scene_paths,
        stage_name="filter_paths_by_extrinsics_quality",
        require_cache=True,
        allow_streaming=allow_gcs_streaming,
    )

    kept_paths = []
    rejected_count = 0
    errors = []

    for scene_path in tqdm(scene_paths, desc="Filtering by extrinsics quality"):
        try:
            uuid = get_uuid(scene_path)
        except Exception as exc:
            errors.append(f"{scene_path}: failed to extract uuid ({exc})")
            continue

        cameras_json_path = os.path.join(output_dir, "cameras", f"{uuid}_cameras.json")
        if not os.path.exists(cameras_json_path):
            errors.append(f"{scene_path} ({uuid}): missing cameras JSON: {cameras_json_path}")
            continue

        try:
            with open(cameras_json_path, "r") as f:
                camera_data = json.load(f)
        except Exception as exc:
            errors.append(f"{scene_path} ({uuid}): failed to read JSON ({exc})")
            continue

        # Reject scenes with error_info
        if "error_info" in camera_data:
            rejected_count += 1
            continue

        # Check optimization_summary.final_loss
        optimization_summary = camera_data.get("optimization_summary")
        if not isinstance(optimization_summary, dict):
            errors.append(f"{scene_path} ({uuid}): missing optimization_summary")
            continue
        if "final_loss" not in optimization_summary:
            errors.append(f"{scene_path} ({uuid}): missing final_loss")
            continue

        try:
            final_loss = float(optimization_summary["final_loss"])
        except Exception as exc:
            errors.append(f"{scene_path} ({uuid}): invalid final_loss={optimization_summary['final_loss']!r}")
            continue

        if final_loss < max_final_loss:
            kept_paths.append(scene_path)
        else:
            rejected_count += 1

    # Report errors
    if errors:
        max_show = 20
        shown = errors[:max_show]
        more = len(errors) - len(shown)
        msg = "\n".join(shown)
        if more > 0:
            msg += f"\n... and {more} more error(s)"
        raise RuntimeError(
            f"Failed to build filtered scene list due to {len(errors)} error(s):\n{msg}"
        )

    # Write output
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w") as f:
        for p in kept_paths:
            f.write(f"{p}\n")

    print(
        f"\n✅ Filtered scene list written to {output_path}:\n"
        f"   kept={len(kept_paths)}, rejected={rejected_count}, "
        f"total={len(scene_paths)}, threshold(final_loss < {max_final_loss})"
    )
    return kept_paths

print("✅ filter_paths_by_extrinsics_quality defined")

In [ ]:
# ============================================================
# Stage A — Run the filter
# ============================================================
kept_paths = filter_paths_by_extrinsics_quality(
    input_path=INPUT_SCENE_LIST,
    output_path=FILTERED_OUTPUT_PATH,
    output_dir=DROID_ROOT,
    max_final_loss=MAX_FINAL_LOSS,
    scene_path_prefix=SCENE_PATH_PREFIX,
    allow_gcs_streaming=ALLOW_GCS_STREAMING,
)

print(f"\n📋 Preview of kept paths ({min(5, len(kept_paths))} of {len(kept_paths)}):")
for p in kept_paths[:5]:
    print(f"  {p}")

---
---
# Stage B — Compute 2D Flows (CoTracker)

This stage runs CoTracker on each filtered scene to produce 2D point tracks across video clips.

**Pipeline overview:**
1. Load RGB frames from SVO + proprioception data
2. Downsample temporally (`time_skip_ratio=2`) and spatially (`downscale_ratio=0.5`)
3. Slice into overlapping clips (`frames_per_clip=11`, `skip_every=5`)
4. Filter clips by end-effector motion (discard static clips)
5. Run CoTracker on each clip → 2D tracks `(T, N, 2)` + visibility `(T, N)`
6. Sample RGB colors at tracked locations
7. Save per-scene `<uuid>_2d_flows.h5`

**Source:** `real/compute_2d_flows.py` (class `TrackCacheRunner`, extends `Flow2DTracker` from `real/flow_2d.py`)

### Stage B — Configuration

In [ ]:
# ============================================================
# Stage B Configuration — edit these values
# ============================================================
import os

# Use the filtered list from Stage A (or set to droid_paths.txt to skip filtering)
DROID_FLOW_INPUT = FILTERED_OUTPUT_PATH
# DROID_FLOW_INPUT = os.path.join(COLAB_ROOT, "real", "droid_paths.txt")

COTRACKER_CKPT_PATH = os.path.join(COLAB_ROOT, "checkpoints", "cotracker", "scaled_online.pth")

# Processing params
TIME_SKIP_RATIO   = 2       # Temporal downsampling: keep every Nth frame
DOWNSCALE_RATIO   = 0.5     # Spatial downsampling for tracking
FRAMES_PER_CLIP   = 11      # Number of frames per clip
SKIP_EVERY        = 5       # Stride between clip start indices
SPACE_SKIP_RATIO  = 2       # Spatial skip ratio for tracking grid

# EE motion thresholds for clip filtering
EE_POS_THRESHOLD  = 0.20    # meters
EE_ROT_THRESHOLD  = np.deg2rad(90)  # radians
GRIPPER_THRESHOLD = 0.1
GRIPPER_CLOSED_EE_POS_THRESHOLD = 0.10
GRIPPER_CLOSED_EE_ROT_THRESHOLD = np.deg2rad(60)

# Distributed processing (single worker in Colab)
RANK       = 0
WORLD_SIZE = 1
RANDOM_SEED = 42
MAX_SCENES = None  # Set to e.g. 5 for quick testing
SKIP_PROCESSED = True

print(f"DROID_FLOW_INPUT:  {DROID_FLOW_INPUT}")
print(f"CoTracker ckpt:    {COTRACKER_CKPT_PATH}")
print(f"Output dir:        {DROID_ROOT}")

### Stage B — Define: `Flow2DTracker` (CoTracker wrapper + clip slicing)

In [ ]:
# ============================================================
# Flow2DTracker — CoTracker-based 2D tracking with clip management
# ============================================================
import inspect
import random
from typing import Dict, Tuple, Any
import torch

DEFAULT_DEVICE = (
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)

class Flow2DTracker:
    def __init__(self, cotracker_ckpt_path=None):
        self.cotracker_ckpt_path = cotracker_ckpt_path or COTRACKER_CKPT_PATH
        self.tracker = None
        self._legacy_skip_ratio_supported = None

    def _load_tracker(self):
        assert os.path.exists(self.cotracker_ckpt_path), (
            f"CoTracker checkpoint not found: {self.cotracker_ckpt_path}"
        )
        from cotracker.predictor import CoTrackerPredictor
        print(f"[{get_time_str()}] Loading CoTracker model...")
        tracker = CoTrackerPredictor(
            checkpoint=self.cotracker_ckpt_path,
            offline=True,
            window_len=16,
            v2=False,
        )
        tracker = tracker.to(DEFAULT_DEVICE).eval()
        print(f"[{get_time_str()}] Model loaded on {DEFAULT_DEVICE}.")
        return tracker

    def slice_video_clips(self, data_dict, proprio_dict, frames_per_clip=16, skip_every=1):
        """
        Chunk each camera's video into overlapping clips of length frames_per_clip.
        Clips are sampled every skip_every frames; sequences shorter than
        frames_per_clip are padded with the last frame.
        """
        print(f"[{get_time_str()}] Chunking videos into overlapping clips...")
        T = len(next(iter(data_dict.values()))["rgb"])
        start_indices = range(0, max(1, T - frames_per_clip + 1), skip_every)

        sliced_data_dict = {}
        for camera_serial in tqdm(data_dict, desc="Chunking camera videos"):
            sliced_data_dict[camera_serial] = {}
            for start in start_indices:
                end = min(start + frames_per_clip, T)
                clip_key = f"{start}:{end}"
                sliced_data_dict[camera_serial][clip_key] = {}
                for modality in ["intrinsic", "extrinsic"]:
                    if modality in data_dict[camera_serial]:
                        sliced_data_dict[camera_serial][modality] = data_dict[camera_serial][modality]
                for modality in data_dict[camera_serial]:
                    if modality in ["intrinsic", "extrinsic", "baseline"]: continue
                    data_slice = data_dict[camera_serial][modality][start:end]
                    if len(data_slice) < frames_per_clip:
                        pad_shape = list(data_slice.shape)
                        pad_shape[0] = frames_per_clip
                        padded = np.zeros(pad_shape, dtype=data_slice.dtype)
                        padded[:len(data_slice)] = data_slice
                        for i in range(len(data_slice), frames_per_clip):
                            padded[i] = data_slice[-1]
                        sliced_data_dict[camera_serial][clip_key][modality] = padded
                    else:
                        sliced_data_dict[camera_serial][clip_key][modality] = data_slice

        sliced_proprio_dict = {}
        for start in start_indices:
            end = min(start + frames_per_clip, T)
            clip_key = f"{start}:{end}"
            sliced_proprio_dict[clip_key] = {}
            for modality in proprio_dict:
                data_slice = proprio_dict[modality][start:end]
                if len(data_slice) < frames_per_clip:
                    if isinstance(data_slice, np.ndarray):
                        pad_shape = list(data_slice.shape)
                        pad_shape[0] = frames_per_clip
                        padded = np.zeros(pad_shape, dtype=data_slice.dtype)
                        padded[:len(data_slice)] = data_slice
                        for i in range(len(data_slice), frames_per_clip):
                            padded[i] = data_slice[-1]
                    else:
                        padded = data_slice + [data_slice[-1]] * (frames_per_clip - len(data_slice))
                    sliced_proprio_dict[clip_key][modality] = padded
                else:
                    sliced_proprio_dict[clip_key][modality] = data_slice
        return sliced_data_dict, sliced_proprio_dict

    def filter_clips_by_ee_motion(
        self, sliced_data_dict, sliced_proprio_dict,
        ee_pos_threshold=0.005, ee_rot_threshold=0.1,
        gripper_threshold=0.1,
        gripper_closed_ee_pos_threshold=0.002,
        gripper_closed_ee_rot_threshold=0.05,
    ):
        """Filter clips based on end-effector motion. Clips with gripper changes are always kept."""
        clip_keys = [key for key in sliced_proprio_dict.keys() if ":" in key]
        initial_count = len(clip_keys)
        valid_clips = []

        for clip_key in clip_keys:
            gripper_poses = sliced_proprio_dict[clip_key]["gripper_pose"]
            gripper_poses = convert_pose_quat2mat(gripper_poses)
            gripper_positions = sliced_proprio_dict[clip_key]["gripper_positions"]
            gripper_open = gripper_positions < gripper_threshold
            has_gripper_change = False
            if len(gripper_open) > 1:
                has_gripper_change = np.any(np.diff(gripper_open.astype(int)) != 0)

            pose_errors = []
            for i in range(len(gripper_poses) - 1):
                pose_errors.append(get_pose_error(gripper_poses[i + 1], gripper_poses[i]))
            pose_errors = np.array(pose_errors)
            total_pos = np.sum(np.linalg.norm(pose_errors[:, :3], axis=1))
            total_rot = np.sum(np.linalg.norm(pose_errors[:, 3:], axis=1))

            if has_gripper_change:
                valid_clips.append(clip_key)
            else:
                is_closed = np.mean(gripper_open) < 0.5
                pos_t = gripper_closed_ee_pos_threshold if is_closed else ee_pos_threshold
                rot_t = gripper_closed_ee_rot_threshold if is_closed else ee_rot_threshold
                if total_pos >= pos_t or total_rot >= rot_t:
                    valid_clips.append(clip_key)

        filtered_pct = (initial_count - len(valid_clips)) / max(1, initial_count) * 100
        print(f"[{get_time_str()}] EE motion filter: kept {len(valid_clips)}/{initial_count} clips ({filtered_pct:.1f}% filtered)")

        filtered_proprio = {k: sliced_proprio_dict[k] for k in valid_clips}
        filtered_data = {}
        for cam in sliced_data_dict:
            filtered_data[cam] = {k: v for k, v in sliced_data_dict[cam].items() if ':' not in k}
            for ck in valid_clips:
                if ck in sliced_data_dict[cam]:
                    filtered_data[cam][ck] = sliced_data_dict[cam][ck]
        return filtered_data, filtered_proprio

    def _tracker_supports_legacy_skip_ratio(self):
        if self.tracker is None: return False
        if self._legacy_skip_ratio_supported is None:
            try:
                sig = inspect.signature(self.tracker.forward)
                self._legacy_skip_ratio_supported = "skip_ratio" in sig.parameters
            except (TypeError, ValueError):
                self._legacy_skip_ratio_supported = False
        return bool(self._legacy_skip_ratio_supported)

    @torch.inference_mode()
    @torch.cuda.amp.autocast(True)
    def _tracker_inference(self, rgb_seq, space_skip_ratio=4):
        """Run CoTracker on an RGB clip (T, H, W, 3) uint8."""
        if self.tracker is None:
            self.tracker = self._load_tracker()
        rgb_tensor = torch.from_numpy(rgb_seq).float()
        rgb_tensor = rgb_tensor.permute(0, 3, 1, 2).unsqueeze(0).to(DEFAULT_DEVICE)

        if self._tracker_supports_legacy_skip_ratio():
            pred_tracks, pred_visibility = self.tracker(
                rgb_tensor, grid_size=0, grid_query_frame=0,
                backward_tracking=True, skip_ratio=space_skip_ratio, show_progress=False,
            )
            return pred_tracks[0].cpu().numpy(), pred_visibility[0, :, :].cpu().numpy()

        # Fallback: explicit grid queries
        _, T_clip, _, H, W = rgb_tensor.shape
        ys = torch.arange(0, H, space_skip_ratio, device=rgb_tensor.device, dtype=torch.float32)
        xs = torch.arange(0, W, space_skip_ratio, device=rgb_tensor.device, dtype=torch.float32)
        grid_y, grid_x = torch.meshgrid(ys, xs, indexing="ij")
        x_flat = grid_x.reshape(-1)
        y_flat = grid_y.reshape(-1)
        t_flat = torch.zeros_like(x_flat)
        queries = torch.stack([t_flat, x_flat, y_flat], dim=1).unsqueeze(0)

        total_queries = queries.shape[1]
        chunk_size = 8192
        pred_tracks = np.zeros((T_clip, total_queries, 2), dtype=np.float32)
        pred_visibility = np.zeros((T_clip, total_queries), dtype=bool)
        for start in range(0, total_queries, chunk_size):
            end = min(start + chunk_size, total_queries)
            tracks_chunk, vis_chunk = self.tracker(
                rgb_tensor, queries=queries[:, start:end], backward_tracking=True,
            )
            pred_tracks[:, start:end] = tracks_chunk[0].cpu().numpy()
            pred_visibility[:, start:end] = vis_chunk[0, :, :].cpu().numpy()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        return pred_tracks, pred_visibility

print(f"✅ Flow2DTracker defined (device: {DEFAULT_DEVICE})")

### Stage B — Define: `TrackCacheRunner` (orchestrates per-scene 2D tracking)

In [ ]:
# ============================================================
# TrackCacheRunner — Runs maskless 2D tracking over DROID scenes
# ============================================================
import time as _time

class TrackCacheRunner(Flow2DTracker):
    """
    Runs maskless 2D tracking over DROID scenes, slices into clips,
    and writes a compact tracks cache per scene for later 3D lifting.

    Notes:
    - Does not apply robot/workspace masks during tracking (maskless).
    - Stores per-clip first RGB frame for compatibility in later conversions.
    """

    def __init__(self, cotracker_ckpt_path=None):
        super().__init__(cotracker_ckpt_path=cotracker_ckpt_path)

    def process(
        self, scene_path, save_dir,
        time_skip_ratio=2, downscale_ratio=0.5,
        frames_per_clip=11, skip_every=5, space_skip_ratio=2,
        skip_processed=True,
        ee_pos_threshold=0.20, ee_rot_threshold=np.deg2rad(90),
        gripper_threshold=0.1,
        gripper_closed_ee_pos_threshold=0.10,
        gripper_closed_ee_rot_threshold=np.deg2rad(60),
    ):
        self.output_dir = save_dir
        self.scene_path = scene_path
        self.uuid = get_uuid(scene_path)
        self.save_dir = os.path.join(self.output_dir, '2d_flows')
        self.cache_h5_path = os.path.join(self.save_dir, f'{self.uuid}_2d_flows.h5')
        self.cache_h5_tmp_path = self.cache_h5_path + ".tmp"

        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(self.save_dir, exist_ok=True)

        # Resume/skip logic
        def _assess_state(path):
            if not os.path.exists(path): return 'absent'
            with h5py.File(path, 'r') as f:
                return 'complete' if bool(f.attrs.get('write_complete', False)) else 'incomplete'

        if skip_processed:
            state = _assess_state(self.cache_h5_path)
            if state == 'complete':
                print(f"Skipping scene {scene_path} - 2d flows complete")
                return True

        if os.path.exists(self.cache_h5_tmp_path):
            os.remove(self.cache_h5_tmp_path)
        self.tracker = self._load_tracker()

        # Load data
        print(f"[{get_time_str()}] Processing scene: {scene_path}")
        data_dict = gather_data_dict(
            scene_path, downscale_ratio=downscale_ratio,
            include_stereo=False, include_depth=False,
        )
        proprio_dict = gather_trajectory(scene_path)

        # Canonical timestamps + temporal downsampling
        first_cam = list(data_dict.keys())[0]
        self.canonical_timestamps = data_dict[first_cam]['timestamps']
        if time_skip_ratio > 1:
            self.canonical_timestamps = self.canonical_timestamps[::time_skip_ratio]
        self.T = len(self.canonical_timestamps)
        data_dict = filter_by_timestamps(data_dict, self.canonical_timestamps, scene_path)
        proprio_dict = filter_by_timestamps(proprio_dict, self.canonical_timestamps, scene_path, is_proprio=True)

        # Slice + filter clips by EE motion
        sliced_data_dict, sliced_proprio_dict = self.slice_video_clips(
            data_dict, proprio_dict, frames_per_clip=frames_per_clip, skip_every=skip_every
        )
        sliced_data_dict, sliced_proprio_dict = self.filter_clips_by_ee_motion(
            sliced_data_dict, sliced_proprio_dict,
            ee_pos_threshold=ee_pos_threshold, ee_rot_threshold=ee_rot_threshold,
            gripper_threshold=gripper_threshold,
            gripper_closed_ee_pos_threshold=gripper_closed_ee_pos_threshold,
            gripper_closed_ee_rot_threshold=gripper_closed_ee_rot_threshold,
        )

        # Run tracker on each clip
        print(f"[{get_time_str()}] Running tracker on clips...")
        cache_payload = {
            'uuid': self.uuid, 'scene_path': self.scene_path,
            'canonical_timestamps': self.canonical_timestamps,
            'frames_per_clip': frames_per_clip, 'skip_every': skip_every,
            'time_skip_ratio': time_skip_ratio, 'downscale_ratio': downscale_ratio,
            'space_skip_ratio': space_skip_ratio,
            'tracker_type': 'cotracker', 'tracker_ckpt': self.cotracker_ckpt_path,
            'ee_pos_threshold': float(ee_pos_threshold),
            'ee_rot_threshold': float(ee_rot_threshold),
            'gripper_closed_ee_pos_threshold': float(gripper_closed_ee_pos_threshold),
            'gripper_closed_ee_rot_threshold': float(gripper_closed_ee_rot_threshold),
            'gripper_threshold': float(gripper_threshold),
        }

        payload_by_camera = {}
        for camera_serial, camera_data in sliced_data_dict.items():
            intrinsic = camera_data['intrinsic']
            camera_type = 'ext'

            for clip_key, clip_rec in camera_data.items():
                if ':' not in clip_key: continue
                rgb_seq = clip_rec['rgb']  # (T, H, W, 3)
                T_clip, H, W, _ = rgb_seq.shape

                pred_tracks, pred_visibility = self._tracker_inference(
                    rgb_seq, space_skip_ratio=space_skip_ratio,
                )

                # OOB invalidation, color sampling
                tracks_rounded = np.round(pred_tracks).astype(int)
                oob = (pred_tracks[...,0]<0)|(pred_tracks[...,0]>=W)|(pred_tracks[...,1]<0)|(pred_tracks[...,1]>=H)
                pred_visibility[oob] = 0
                x_coords = np.clip(tracks_rounded[..., 0], 0, W - 1)
                y_coords = np.clip(tracks_rounded[..., 1], 0, H - 1)
                t_idx = np.arange(T_clip)[:, None]
                flow_colors = rgb_seq[t_idx, y_coords, x_coords].astype(np.uint8)

                cam_key = f"{camera_serial}+{camera_type}"
                if cam_key not in payload_by_camera:
                    payload_by_camera[cam_key] = {'intrinsic': intrinsic, 'clips': {}}

                payload_by_camera[cam_key]['clips'][clip_key] = {
                    'H': int(H), 'W': int(W),
                    'flows_2d_xy': pred_tracks.astype(np.float32),
                    'flows_2d_visibility': pred_visibility.astype(bool),
                    'flow_colors': flow_colors.astype(np.uint8),
                    'first_frame_rgb': rgb_seq[0].astype(np.uint8),
                }

        # Proprio per-clip
        proprio_by_clip = {}
        for clip_key, rec in sliced_proprio_dict.items():
            if ':' not in clip_key: continue
            proprio_by_clip[clip_key] = {
                'joint_positions': rec['joint_positions'].astype(np.float32),
                'joint_velocities': rec['joint_velocities'].astype(np.float32),
                'joint_torques': rec['joint_torques'].astype(np.float32),
                'gripper_positions': rec['gripper_positions'].astype(np.float32),
                'gripper_pose': rec['gripper_pose'].astype(np.float32),
            }

        # Save
        self.store_2d_flows(self.cache_h5_path, cache_payload, payload_by_camera, proprio_by_clip)
        print(f"[{get_time_str()}] 2d flows saved to {self.cache_h5_path}")
        return True

    def store_2d_flows(self, output_path, meta, cams_dict, proprio_dict):
        """Write 2D flows cache to HDF5 (atomic write via temp file)."""
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        tmp_path = output_path + ".tmp"
        with h5py.File(tmp_path, 'w') as f:
            f.attrs['uuid'] = meta['uuid']
            f.attrs['scene_path'] = meta['scene_path']
            f.attrs['creation_time'] = _time.strftime("%Y%m%d_%H%M%S")
            f.attrs['canonical_timestamps'] = json.dumps(list(map(int, meta['canonical_timestamps'])))
            f.attrs['frames_per_clip'] = int(meta['frames_per_clip'])
            f.attrs['skip_every'] = int(meta['skip_every'])
            f.attrs['time_skip_ratio'] = int(meta['time_skip_ratio'])
            f.attrs['downscale_ratio'] = float(meta['downscale_ratio'])
            f.attrs['space_skip_ratio'] = int(meta['space_skip_ratio'])
            f.attrs['tracker_type'] = meta['tracker_type']
            f.attrs['tracker_ckpt'] = meta['tracker_ckpt']
            f.attrs['ee_pos_threshold'] = float(meta['ee_pos_threshold'])
            f.attrs['ee_rot_threshold'] = float(meta['ee_rot_threshold'])
            f.attrs['gripper_closed_ee_pos_threshold'] = float(meta['gripper_closed_ee_pos_threshold'])
            f.attrs['gripper_closed_ee_rot_threshold'] = float(meta['gripper_closed_ee_rot_threshold'])
            f.attrs['gripper_threshold'] = float(meta['gripper_threshold'])
            f.attrs['write_complete'] = False

            cams_group = f.create_group('cameras')
            for cam_key, cam_payload in cams_dict.items():
                g_cam = cams_group.create_group(cam_key)
                g_cam.create_dataset('intrinsic', data=cam_payload['intrinsic'].astype(np.float32))
                g_clips = g_cam.create_group('clips')
                for clip_key, clip_data in cam_payload['clips'].items():
                    g_clip = g_clips.create_group(clip_key)
                    g_clip.attrs['H'] = clip_data['H']
                    g_clip.attrs['W'] = clip_data['W']
                    g_clip.create_dataset('flows_2d_xy', data=clip_data['flows_2d_xy'])
                    g_clip.create_dataset('flows_2d_visibility', data=clip_data['flows_2d_visibility'])
                    g_clip.create_dataset('flow_colors', data=clip_data['flow_colors'])
                    save_rgb_as_jpeg_in_h5(g_clip, 'initial_rgb', clip_data['first_frame_rgb'])

            proprio_group = f.create_group('proprio')
            for clip_key, pr in proprio_dict.items():
                g_p = proprio_group.create_group(clip_key)
                for k, v in pr.items():
                    g_p.create_dataset(k, data=v)

            f.attrs['write_complete'] = True
        os.replace(tmp_path, output_path)

print("✅ TrackCacheRunner defined")

In [ ]:
# ============================================================
# Stage B — Run 2D flow computation
# ============================================================
import random

# Read scene paths from the filtered list
with open(DROID_FLOW_INPUT, 'r') as f:
    flow_scene_paths = [line.strip() for line in f if line.strip()]

enforce_gcs_cache_policy(
    flow_scene_paths,
    stage_name='compute_2d_flows',
    require_cache=True,
    allow_streaming=ALLOW_GCS_STREAMING,
)

random.seed(RANDOM_SEED)
random.shuffle(flow_scene_paths)
if MAX_SCENES is not None and MAX_SCENES > 0:
    flow_scene_paths = flow_scene_paths[:MAX_SCENES]

# Distribute across workers
worker_scenes = flow_scene_paths[RANK::WORLD_SIZE]
print(f"Worker {RANK}/{WORLD_SIZE} processing {len(worker_scenes)} scenes for 2D flows")

# Run
runner = TrackCacheRunner(cotracker_ckpt_path=COTRACKER_CKPT_PATH)

pbar = tqdm(worker_scenes, desc=f"worker {RANK} 2D flows", unit="scene", dynamic_ncols=True)
for scene_path in pbar:
    tqdm.write(f"[{get_time_str()}] Processing: {scene_path}")
    ok = runner.process(
        scene_path, save_dir=DROID_ROOT,
        time_skip_ratio=TIME_SKIP_RATIO, downscale_ratio=DOWNSCALE_RATIO,
        frames_per_clip=FRAMES_PER_CLIP, skip_every=SKIP_EVERY,
        space_skip_ratio=SPACE_SKIP_RATIO, skip_processed=SKIP_PROCESSED,
        ee_pos_threshold=EE_POS_THRESHOLD, ee_rot_threshold=EE_ROT_THRESHOLD,
        gripper_threshold=GRIPPER_THRESHOLD,
        gripper_closed_ee_pos_threshold=GRIPPER_CLOSED_EE_POS_THRESHOLD,
        gripper_closed_ee_rot_threshold=GRIPPER_CLOSED_EE_ROT_THRESHOLD,
    )
    tqdm.write(f"[{get_time_str()}] Done: {get_uuid(scene_path)} status={ok}")

print(f"\n✅ 2D flow computation complete!")

---
---
# Stage C — Convert 2D Flows to 3D

This stage loads cached 2D flows + depth + optimized extrinsics, and produces the final 3D flow data:

1. Load 2D flows cache (`<uuid>_2d_flows.h5`)
2. Load optimized extrinsics (`cameras/<uuid>_cameras.json`)
3. Load depth maps (`depth/<uuid>_depth.h5`)
4. For each clip:
   - Scale 2D track coordinates to depth resolution
   - Sample depth at track locations (streaming, low memory)
   - Compute workspace mask (convex hull of projected workspace bounds)
   - Compute robot mask (FK + projected mesh points)
   - Apply seed-frame keep mask (workspace ∩ ¬robot)
   - Backproject 2D tracks → 3D using depth + intrinsics + cam2world
   - Estimate normals (Open3D CPU)
5. Filter by relaxed 3D workspace bounds
6. Remove outlier flows (DBSCAN-based)
7. Write final `<uuid>_flows.h5`

**Source:** `real/convert_2d_flows_to_3d.py` (class `Flow2DTo3DConverter`)

### Stage C — Configuration

In [ ]:
# ============================================================
# Stage C Configuration — edit these values
# ============================================================

EXTRINSICS_SOURCE = 'optimized'  # 'optimized' or 'vggt'

# URDF for robot FK (needed for robot masking)
URDF_PATH = os.path.join(COLAB_ROOT, "assets", "franka_description", "franka_panda_robotiq_2f85.urdf")

# Outlier removal parameters
OUTLIER_EPS_VALUES    = [0.02, 0.05]  # DBSCAN eps values
OUTLIER_MIN_POINTS    = 5
OUTLIER_FRAMES_RATIO  = 0.2
OUTLIER_SCOPE         = 'global'  # 'global', 'per_camera', or 'none'

# Seed mask mode
SEED_MASK_MODE = 'auto'  # 'auto', 'workspace_and_not_robot', 'workspace_only', 'none'

# Optional
ROBOT_MASK_SEED = None
SCALE_INTRINSICS_TO_DEPTH = False

print(f"Extrinsics source: {EXTRINSICS_SOURCE}")
print(f"URDF path:         {URDF_PATH}")
print(f"Output dir:        {DROID_ROOT}")

### Stage C — Define: `flow_postprocessing` (outlier removal)

In [ ]:
# ============================================================
# flow_postprocessing — DBSCAN-based outlier removal for 3D flows
# ============================================================
from sklearn.cluster import DBSCAN

def remove_outlier_flows(
    trajectories, visibility_mask=None, depth_valid_mask=None,
    eps_values=(0.02, 0.05), min_points=5, min_outlier_frames_ratio=0.2,
):
    """
    Multi-eps DBSCAN outlier removal on 3D trajectories.

    Args:
        trajectories: (T, N, 3) 3D point trajectories
        visibility_mask: (T, N) bool — tracker visibility
        depth_valid_mask: (T, N) bool — depth validity
        eps_values: list of DBSCAN eps to try
        min_points: DBSCAN min_samples
        min_outlier_frames_ratio: fraction of frames a point must be outlier

    Returns:
        trajectories, visibility_mask, depth_valid_mask, outlier_indices
    """
    T, N, _ = trajectories.shape
    if N == 0:
        return trajectories, visibility_mask, depth_valid_mask, np.array([], dtype=int)

    combined_mask = np.ones((T, N), dtype=bool)
    if visibility_mask is not None:
        combined_mask &= visibility_mask
    if depth_valid_mask is not None:
        combined_mask &= depth_valid_mask

    outlier_counts = np.zeros(N, dtype=int)
    valid_frame_counts = np.zeros(N, dtype=int)

    for t in range(T):
        valid = combined_mask[t]
        valid_indices = np.where(valid)[0]
        if len(valid_indices) < min_points + 1:
            continue
        valid_frame_counts[valid_indices] += 1
        pts = trajectories[t, valid_indices]

        is_outlier_any_eps = np.zeros(len(valid_indices), dtype=bool)
        for eps in eps_values:
            db = DBSCAN(eps=eps, min_samples=min_points)
            labels = db.fit_predict(pts)
            is_outlier_any_eps |= (labels == -1)
        outlier_counts[valid_indices] += is_outlier_any_eps.astype(int)

    # A point is a global outlier if it's outlier in > min_outlier_frames_ratio of its valid frames
    safe_counts = np.maximum(valid_frame_counts, 1)
    outlier_ratio = outlier_counts / safe_counts
    global_outliers = np.where(outlier_ratio > min_outlier_frames_ratio)[0]

    return trajectories, visibility_mask, depth_valid_mask, global_outliers

print("✅ remove_outlier_flows defined")

### Stage C — Define: `RobotSampler` (FK + mesh sampling for masks)

In [ ]:
# ============================================================
# RobotSampler — Forward Kinematics for DROID robot mesh
# ============================================================
import numpy as np
if not hasattr(np, 'float'):
    np.float = float  # urdfpy compatibility

import urdfpy

class RobotSampler:
    """Minimal robot FK wrapper for building robot masks."""
    def __init__(self, domain='droid', urdf_path=None):
        self.domain = domain
        if urdf_path is None:
            urdf_path = URDF_PATH
        if not os.path.isabs(urdf_path):
            urdf_path = os.path.normpath(os.path.join(REPO_ROOT, urdf_path))
        assert os.path.exists(urdf_path), f"URDF not found: {urdf_path}"
        self.robot_urdf = urdfpy.URDF.load(urdf_path)
        print(f"RobotSampler: loaded URDF from {urdf_path}")

    def forward_kinematics(self, cfg):
        """Run visual trimesh FK with the given joint configuration dict."""
        return self.robot_urdf.visual_trimesh_fk(cfg=cfg)

print("✅ RobotSampler defined")

### Stage C — Define: `data_contract` (expected shapes)

In [ ]:
# ============================================================
# data_contract — Expected camera payload shapes
# ============================================================
EXPECTED_CAMERA_PAYLOAD_SHAPES = {
    "initial_depth": (360, 640),
    "initial_rgb": (360, 640, 3),
}

QUANTIZED_NORMALS_DTYPE = np.int8

def _quantize_unit_normals_to_int8(normals):
    normals_f32 = np.asarray(normals, dtype=np.float32)
    clipped = np.clip(normals_f32, -1.0, 1.0)
    return np.rint(clipped * 127.0).astype(QUANTIZED_NORMALS_DTYPE)

print("✅ data_contract defined")

### Stage C — Define: `Flow2DTo3DConverter`

In [ ]:
# ============================================================
# Flow2DTo3DConverter — CPU-only 2D→3D flow conversion
# ============================================================
import time as _time

MIN_DEPTH_M = 0.0
MAX_DEPTH_M = 4.0

def _resize_to_contract(image, target_shape, interpolation):
    """Resize image to target (H, W) if needed."""
    target_h, target_w = target_shape
    if image.shape[:2] == (target_h, target_w):
        return image
    return cv2.resize(image, (target_w, target_h), interpolation=interpolation)

def _scale_intrinsic_for_resolution(intrinsic, source_hw, target_hw):
    intr = np.asarray(intrinsic, dtype=np.float32).copy()
    src_h, src_w = int(source_hw[0]), int(source_hw[1])
    dst_h, dst_w = int(target_hw[0]), int(target_hw[1])
    if (src_h, src_w) == (dst_h, dst_w): return intr
    sx, sy = float(dst_w)/float(src_w), float(dst_h)/float(src_h)
    intr[0,0] *= sx; intr[0,2] *= sx
    intr[1,1] *= sy; intr[1,2] *= sy
    return intr


class Flow2DTo3DConverter:
    """
    CPU-only converter that:
    1. Loads cached 2D flows + metadata
    2. Loads extrinsics (optimized or VGGT) and depth
    3. Applies workspace + robot masks (post-hoc, frame-0 only)
    4. Lifts 2D tracks to 3D via depth/intrinsics
    5. Estimates normals (Open3D)
    6. Removes outliers (DBSCAN)
    7. Writes final flows H5
    """

    def __init__(self, urdf_path=None, robot_mask_seed=None):
        if urdf_path is None:
            urdf_path = URDF_PATH
        if not os.path.isabs(urdf_path):
            urdf_path = os.path.normpath(os.path.join(os.path.dirname(os.path.abspath('.')), urdf_path))
        self.robot_sampler = RobotSampler(domain='droid', urdf_path=urdf_path)
        self.robot_mask_seed = robot_mask_seed
        self.mesh_presampled_points = {}
        self.mesh_presampled_ready = False
        self.domain = 'droid'

    def _load_2d_flows(self, cache_path):
        with h5py.File(cache_path, 'r') as f:
            if not bool(f.attrs.get('write_complete', False)):
                raise RuntimeError(f"2d flows cache incomplete: {cache_path}")
            uuid = f.attrs['uuid']
            scene_path = f.attrs['scene_path']
            canonical_timestamps = np.array(json.loads(f.attrs['canonical_timestamps']))
            frames_per_clip = int(f.attrs['frames_per_clip'])
            skip_every = int(f.attrs['skip_every'])
            downscale_ratio = float(f.attrs['downscale_ratio'])
            space_skip_ratio = int(f.attrs['space_skip_ratio'])
            tracker_type = f.attrs['tracker_type']
            def _s(v): return v.decode('utf-8') if isinstance(v, (bytes,bytearray)) else str(v)
            tracking_mask_mode = _s(f.attrs.get('tracking_mask_mode', 'none'))
            ee_pos_threshold = float(f.attrs['ee_pos_threshold'])
            ee_rot_threshold = float(f.attrs['ee_rot_threshold'])
            gripper_closed_ee_pos_threshold = float(f.attrs['gripper_closed_ee_pos_threshold'])
            gripper_closed_ee_rot_threshold = float(f.attrs['gripper_closed_ee_rot_threshold'])

            cams = {}
            for cam_key in f['cameras'].keys():
                g_cam = f['cameras'][cam_key]
                intrinsic = np.array(g_cam['intrinsic'])
                clips = {}
                for clip_key in g_cam['clips'].keys():
                    g_clip = g_cam['clips'][clip_key]
                    initial_rgb = load_rgb_from_jpeg_in_h5(g_clip['initial_rgb'])
                    clips[clip_key] = {
                        'H': int(g_clip.attrs['H']), 'W': int(g_clip.attrs['W']),
                        'flows_2d_xy': np.array(g_clip['flows_2d_xy']),
                        'flows_2d_visibility': np.array(g_clip['flows_2d_visibility']).astype(bool),
                        'flow_colors': np.array(g_clip['flow_colors']).astype(np.uint8),
                        'initial_rgb': initial_rgb.astype(np.uint8),
                    }
                cams[cam_key] = {'intrinsic': intrinsic, 'clips': clips}

            proprio = {}
            for clip_key in f['proprio'].keys():
                g_p = f['proprio'][clip_key]
                proprio[clip_key] = {
                    'joint_positions': np.array(g_p['joint_positions']),
                    'joint_velocities': np.array(g_p['joint_velocities']),
                    'joint_torques': np.array(g_p['joint_torques']),
                    'gripper_positions': np.array(g_p['gripper_positions']),
                    'gripper_pose': np.array(g_p['gripper_pose']),
                }

        return {
            'uuid': uuid, 'scene_path': scene_path,
            'canonical_timestamps': canonical_timestamps,
            'frames_per_clip': frames_per_clip, 'skip_every': skip_every,
            'downscale_ratio': downscale_ratio, 'space_skip_ratio': space_skip_ratio,
            'tracker_type': tracker_type,
            'tracking_mask_mode': tracking_mask_mode,
            'ee_pos_threshold': ee_pos_threshold, 'ee_rot_threshold': ee_rot_threshold,
            'gripper_closed_ee_pos_threshold': gripper_closed_ee_pos_threshold,
            'gripper_closed_ee_rot_threshold': gripper_closed_ee_rot_threshold,
            'cameras': cams, 'proprio': proprio,
        }

    def _load_extrinsics(self, output_dir, uuid, camera_serials, source):
        cameras_json = os.path.join(output_dir, 'cameras', f'{uuid}_cameras.json')
        assert os.path.exists(cameras_json), f"Not found: {cameras_json}"
        with open(cameras_json, 'r') as f:
            data = json.load(f)
        key = f"{source}_extrinsics"
        return {s: np.array(data[s][key], dtype=np.float32) for s in camera_serials}

    def _sample_depth_for_tracks(self, depth_ds, depth_ts, clip_ts, x_int, y_int):
        """Stream depth per-frame at track pixel locations (low memory)."""
        Hd, Wd = depth_ds.shape[1:]
        T, N = x_int.shape
        idxs = np.array([int(np.argmin(np.abs(depth_ts - ts))) for ts in clip_ts])
        z = np.zeros((T, N), dtype=np.float32)
        depth_valid = np.zeros((T, N), dtype=bool)
        first_depth_grid = None
        max_mm = int(MAX_DEPTH_M * 1000.0 + 0.5)
        for ti in range(T):
            frame_mm = depth_ds[int(idxs[ti]), ...]
            vals_mm = frame_mm[y_int[ti], x_int[ti]]
            valid = (vals_mm > 0) & (vals_mm <= max_mm)
            depth_valid[ti] = valid
            z[ti] = vals_mm.astype(np.float32) / 1000.0
            if ti == 0:
                frame_m = frame_mm.astype(np.float32) / 1000.0
                frame_valid = (frame_mm > 0) & (frame_mm <= max_mm)
                first_depth_grid = np.where(frame_valid, frame_m, 0.0)[None].astype(np.float32)
        return z, depth_valid, Hd, Wd, first_depth_grid

    def _compute_workspace_mask(self, intrinsic, extrinsic, height, width):
        from scipy.spatial import ConvexHull
        from skimage.draw import polygon
        mask = np.zeros((height, width), dtype=bool)
        pts_px = generate_and_project_workspace_boundary(
            WORKSPACE_BOUNDS_MIN.astype(np.float32), WORKSPACE_BOUNDS_MAX.astype(np.float32),
            100, extrinsic.astype(np.float32), intrinsic.astype(np.float32), width, height,
        )
        if len(pts_px) < 3: return np.ones_like(mask, dtype=bool)
        hull = ConvexHull(pts_px)
        vertices = np.round(pts_px[hull.vertices]).astype(int)
        rr, cc = polygon(vertices[:, 1], vertices[:, 0], shape=(height, width))
        mask[rr, cc] = True
        return mask

    def _presample_mesh_points(self, fk_result, total_samples=100000, gripper_multiplier=2.0):
        self.mesh_presampled_points = {}
        mesh_names, mesh_objs, mesh_areas = [], [], []
        for i, mesh in enumerate(fk_result):
            name = get_mesh_name(mesh, i)
            if mesh.area <= 0: continue
            eff_area = mesh.area
            if 'hand_camera_part' in name.lower(): eff_area *= 1e-6
            mesh_names.append(name); mesh_objs.append(mesh); mesh_areas.append(eff_area)
        if not mesh_names:
            self.mesh_presampled_ready = True; return
        total_area = float(np.sum(mesh_areas))
        rng_state = None
        if self.robot_mask_seed is not None:
            rng_state = np.random.get_state()
            np.random.seed(int(self.robot_mask_seed) % (2**32 - 1))
        try:
            for name, mesh, area in zip(mesh_names, mesh_objs, mesh_areas):
                frac = 0.0 if total_area <= 0 else area / total_area
                n = int(total_samples * frac)
                if any(k in name.lower() for k in ['finger','knuckle','robotiq']):
                    n = int(n * gripper_multiplier)
                n = max(500, n)
                try: pts = mesh.sample(n)
                except Exception:
                    import trimesh; pts, _ = trimesh.sample.sample_surface_even(mesh, n)
                self.mesh_presampled_points[name] = pts.astype(np.float32)
        finally:
            if rng_state is not None: np.random.set_state(rng_state)
        self.mesh_presampled_ready = True

    def _build_robot_mask_frame0(self, H, W, intrinsic, world2cam, joint_positions, gripper_position, downscale_ratio):
        cfg = {'finger_joint': float(gripper_position)}
        for ji in range(7): cfg[f'panda_joint{ji+1}'] = float(joint_positions[ji])
        fk_result = self.robot_sampler.forward_kinematics(cfg)
        if not self.mesh_presampled_ready:
            self._presample_mesh_points(fk_result)
        robot_mask = np.zeros((H, W), dtype=np.uint8)
        std_r = max(1, int(14 * downscale_ratio))
        grip_r = max(1, int(8 * downscale_ratio))
        gripper_kw = ['finger', 'knuckle', 'robotiq']
        for i, mesh in enumerate(fk_result):
            mn = get_mesh_name(mesh, i)
            is_grip = any(k in mn.lower() for k in gripper_kw)
            cr = grip_r if is_grip else std_r
            if mesh.area <= 0 or mn not in self.mesh_presampled_points: continue
            pts3d = self.mesh_presampled_points[mn]
            xf = fk_result[mesh].astype(np.float32)
            pts2d = project_points_to_image(pts3d, xf, world2cam.astype(np.float32), intrinsic.astype(np.float32), W, H)
            if len(pts2d) == 0: continue
            pts2d = np.round(pts2d).astype(int)
            for (x, y) in pts2d:
                if 0 <= x < W and 0 <= y < H:
                    cv2.circle(robot_mask, (x, y), cr, color=1, thickness=-1)
        kernel_sz = max(1, int(20 * downscale_ratio))
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_sz, kernel_sz))
        robot_mask = cv2.morphologyEx(robot_mask, cv2.MORPH_CLOSE, kernel)
        return robot_mask.astype(bool)

    def _estimate_normals(self, pts_world, world2cam):
        import open3d as o3d
        T, N, _ = pts_world.shape
        normals = np.zeros_like(pts_world, dtype=np.float32)
        cam_pos = -world2cam[:3,:3].T @ world2cam[:3,3]
        for ti in range(T):
            pts = pts_world[ti]
            if pts.shape[0] == 0: continue
            pcd = o3d.geometry.PointCloud()
            pcd.points = o3d.utility.Vector3dVector(pts.astype(np.float64))
            pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
            pcd.orient_normals_towards_camera_location(cam_pos.astype(np.float64))
            n = np.asarray(pcd.normals).astype(np.float32)
            normals[ti] = np.where(np.isfinite(n), n, 0.0)
        # Final sign check
        for ti in range(T):
            to_cam = cam_pos[None,:] - pts_world[ti]
            denom = np.clip(np.linalg.norm(to_cam, axis=1, keepdims=True), 1e-10, None)
            dots = (normals[ti] * (to_cam / denom)).sum(axis=1)
            normals[ti, dots < 0] *= -1.0
        return normals

    def _write_h5(self, processed, output_path, ee_pos_threshold, ee_rot_threshold,
                  gripper_closed_ee_pos_threshold=0.002, gripper_closed_ee_rot_threshold=0.05):
        with h5py.File(output_path, 'w') as f:
            f.attrs['uuid'] = processed['uuid']
            f.attrs['creation_time'] = _time.strftime("%Y%m%d_%H%M%S")
            f.attrs['domain'] = self.domain
            f.attrs['scene_path'] = self.scene_path
            metadata = get_metadata(self.scene_path)
            for key in metadata: f.attrs[key] = metadata[key]
            f.attrs['canonical_timestamps'] = str(processed['canonical_timestamps'].tolist())
            f.attrs['ee_pos_threshold'] = ee_pos_threshold
            f.attrs['ee_rot_threshold'] = ee_rot_threshold
            f.attrs['gripper_closed_ee_pos_threshold'] = gripper_closed_ee_pos_threshold
            f.attrs['gripper_closed_ee_rot_threshold'] = gripper_closed_ee_rot_threshold

            for clip_key, clip_data in processed.items():
                if ':' not in clip_key: continue
                clip_group = f.create_group(clip_key)
                if 'joint_positions' in clip_data:
                    clip_group.attrs['demo_length'] = clip_data['joint_positions'].shape[0]
                robot_keys = ['joint_positions','joint_velocities','joint_torques','gripper_open','gripper_positions','gripper_pose']
                for key in robot_keys:
                    if key not in clip_data: continue
                    data = clip_data[key]
                    dtype = np.bool_ if key == 'gripper_open' else np.float32
                    dset = clip_group.create_dataset(key, data=data.astype(dtype))
                    dset.attrs['write_complete'] = True

                camera_prefixes = set(k.split('_scene_flows')[0] for k in clip_data if k.endswith('_scene_flows'))
                for cp in camera_prefixes:
                    cg = clip_group.create_group(f'camera_{cp}')
                    for suf, dt in [('scene_flows',np.float16),('scene_colors',np.uint8),
                                    ('scene_normals',QUANTIZED_NORMALS_DTYPE),
                                    ('scene_visibility',np.bool_),('scene_depth_valid_mask',np.bool_)]:
                        fk = f'{cp}_{suf}'
                        d = clip_data[fk]
                        if suf == 'scene_normals': d = _quantize_unit_normals_to_int8(d)
                        dset = cg.create_dataset(suf, data=d.astype(dt))
                        dset.attrs['write_complete'] = True

                    # Initial RGB/depth (first frame, resized to contract)
                    rgb_src = np.asarray(clip_data[f'{cp}_rgb'][0])
                    depth_src = np.asarray(clip_data[f'{cp}_depth'][0], dtype=np.float32)
                    target_hw = EXPECTED_CAMERA_PAYLOAD_SHAPES['initial_depth']
                    initial_rgb = _resize_to_contract(rgb_src, target_hw, cv2.INTER_AREA)
                    initial_depth = _resize_to_contract(depth_src, target_hw, cv2.INTER_NEAREST)
                    intrinsic_src = np.asarray(clip_data[f'{cp}_intrinsic'], dtype=np.float32)
                    intr = _scale_intrinsic_for_resolution(intrinsic_src, depth_src.shape, initial_depth.shape)
                    extr = np.asarray(clip_data[f'{cp}_extrinsic'], dtype=np.float32)

                    dset = cg.create_dataset('intrinsic', data=intr.astype(np.float32)); dset.attrs['write_complete'] = True
                    dset = cg.create_dataset('extrinsic', data=extr.astype(np.float32)); dset.attrs['write_complete'] = True
                    save_rgb_as_jpeg_in_h5(cg, 'initial_rgb', initial_rgb.astype(np.uint8))
                    save_depth_as_uint16_mm(cg, 'initial_depth', initial_depth)

    def convert_scene(self, cache_path, output_dir, extrinsics_source='optimized',
                      outlier_eps_values=(0.02,0.05), outlier_min_points=5,
                      outlier_frames_ratio=0.2, outlier_scope='global',
                      scale_intrinsics_to_depth=False, seed_mask_mode='auto'):
        cache = self._load_2d_flows(cache_path)
        uuid = cache['uuid']
        scene_path = cache['scene_path']
        timestamps = cache['canonical_timestamps']
        downscale_ratio = cache['downscale_ratio']
        tracking_mask_mode = cache.get('tracking_mask_mode', 'none')

        effective_mask = (
            'none' if (seed_mask_mode == 'auto' and tracking_mask_mode != 'none')
            else ('workspace_and_not_robot' if seed_mask_mode == 'auto' else seed_mask_mode)
        )
        print(f"[{get_time_str()}] Converting {uuid}: effective_seed_mask_mode={effective_mask}")

        depth_h5_path = os.path.join(output_dir, 'depth', f'{uuid}_depth.h5')
        assert os.path.exists(depth_h5_path), f"Depth not found: {depth_h5_path}"

        camera_serials = [k.split('+')[0] for k in cache['cameras'].keys()]
        world2cam_map = self._load_extrinsics(output_dir, uuid, camera_serials, extrinsics_source)

        processed = {'uuid': uuid, 'canonical_timestamps': timestamps}

        for cam_key, cam_payload in cache['cameras'].items():
            camera_serial = cam_key.split('+')[0]
            K = cam_payload['intrinsic']
            world2cam = world2cam_map[camera_serial]
            cam2world = np.linalg.inv(world2cam)

            with h5py.File(depth_h5_path, 'r') as _f:
                _g = _f[cam_key]
                _depth_ds = _g['depth']
                _depth_ts = np.array(_g['timestamps'])
                Hd, Wd = _depth_ds.shape[1:]

                for clip_key, c in cam_payload['clips'].items():
                    start, end = [int(x) for x in clip_key.split(':')]
                    clip_ts = timestamps[start:end]
                    H_rgb, W_rgb = c['H'], c['W']
                    flows_2d = c['flows_2d_xy']
                    vis = c['flows_2d_visibility'].copy()
                    colors = c['flow_colors']

                    # Scale to depth resolution
                    sx, sy = Wd/float(W_rgb), Hd/float(H_rgb)
                    K_lift = K.copy()
                    if scale_intrinsics_to_depth and (abs(sx-1)>1e-6 or abs(sy-1)>1e-6):
                        K_lift[0,0]*=sx; K_lift[0,2]*=sx; K_lift[1,1]*=sy; K_lift[1,2]*=sy
                    td = flows_2d.copy()
                    if abs(sx-1)>1e-6 or abs(sy-1)>1e-6:
                        td[...,0]*=sx; td[...,1]*=sy

                    tr = np.round(td).astype(int)
                    oob = (td[...,0]<0)|(td[...,0]>=Wd)|(td[...,1]<0)|(td[...,1]>=Hd)
                    vis[oob] = False
                    x_int = np.clip(tr[...,0], 0, Wd-1)
                    y_int = np.clip(tr[...,1], 0, Hd-1)

                    T_clip = flows_2d.shape[0]
                    if clip_ts.shape[0] < T_clip:
                        pad_k = T_clip - clip_ts.shape[0]
                        clip_ts = np.concatenate([clip_ts, np.repeat(clip_ts[-1], pad_k)])
                        x_int = np.concatenate([x_int, np.repeat(x_int[-1:], pad_k, axis=0)])
                        y_int = np.concatenate([y_int, np.repeat(y_int[-1:], pad_k, axis=0)])

                    z_tn, depth_valid, _, _, first_depth = self._sample_depth_for_tracks(
                        _depth_ds, _depth_ts, clip_ts, x_int, y_int
                    )

                    # Masks (frame 0)
                    ws_mask = self._compute_workspace_mask(K_lift, world2cam, Hd, Wd)
                    jp = cache['proprio'][clip_key]['joint_positions'][0]
                    gp = cache['proprio'][clip_key]['gripper_positions'][0]
                    rob_mask = self._build_robot_mask_frame0(Hd, Wd, K_lift, world2cam, jp, gp, downscale_ratio)

                    if effective_mask == 'workspace_and_not_robot':
                        keep0 = ws_mask & (~rob_mask)
                    elif effective_mask == 'workspace_only':
                        keep0 = ws_mask
                    else:
                        keep0 = np.ones((Hd, Wd), dtype=bool)
                    keep = keep0[y_int[0], x_int[0]].astype(bool)

                    # Lift to 3D
                    z_lift = z_tn.copy()
                    invalid = (~np.isfinite(z_lift))|(z_lift<MIN_DEPTH_M)|(z_lift>MAX_DEPTH_M)
                    z_lift[invalid] = 0.0
                    fx, fy = K_lift[0,0], K_lift[1,1]
                    cx, cy = K_lift[0,2], K_lift[1,2]
                    X = (x_int - cx)/fx * z_lift
                    Y = (y_int - cy)/fy * z_lift
                    pts_cam = np.stack([X, Y, z_lift, np.ones_like(z_lift)], axis=-1)
                    pts_world = (pts_cam @ cam2world.T)[...,:3]
                    normals = self._estimate_normals(pts_world.astype(np.float32), world2cam.astype(np.float32))

                    # Apply keep mask
                    pts_world = pts_world[:,keep]; normals = normals[:,keep]
                    colors = colors[:,keep]; vis = vis[:,keep]; depth_valid = depth_valid[:,keep]

                    camera_prefix = f"{camera_serial}_ext"
                    if clip_key not in processed:
                        gripper_open = (cache['proprio'][clip_key]['gripper_positions'] < 0.1).astype(bool)
                        if gripper_open.ndim == 1: gripper_open = gripper_open[:, None]
                        processed[clip_key] = {
                            'joint_positions': cache['proprio'][clip_key]['joint_positions'],
                            'joint_velocities': cache['proprio'][clip_key]['joint_velocities'],
                            'joint_torques': cache['proprio'][clip_key]['joint_torques'],
                            'gripper_positions': cache['proprio'][clip_key]['gripper_positions'],
                            'gripper_open': gripper_open,
                            'gripper_pose': cache['proprio'][clip_key]['gripper_pose'],
                        }
                    processed[clip_key][f'{camera_prefix}_scene_flows'] = pts_world.astype(np.float32)
                    processed[clip_key][f'{camera_prefix}_scene_colors'] = colors.astype(np.uint8)
                    processed[clip_key][f'{camera_prefix}_scene_normals'] = normals.astype(np.float32)
                    processed[clip_key][f'{camera_prefix}_scene_visibility'] = vis.astype(bool)
                    processed[clip_key][f'{camera_prefix}_scene_depth_valid_mask'] = depth_valid.astype(bool)
                    processed[clip_key][f'{camera_prefix}_rgb'] = c['initial_rgb'][None].astype(np.uint8)
                    processed[clip_key][f'{camera_prefix}_depth'] = first_depth.astype(np.float32)
                    processed[clip_key][f'{camera_prefix}_intrinsic'] = K_lift.astype(np.float32)
                    processed[clip_key][f'{camera_prefix}_extrinsic'] = world2cam.astype(np.float32)

        # 3D workspace bounds filter
        for clip_key in list(processed.keys()):
            if ':' not in clip_key: continue
            cps = [k.replace('_scene_flows','') for k in processed[clip_key] if k.endswith('_scene_flows')]
            for cp in cps:
                pts = processed[clip_key][f'{cp}_scene_flows']
                xin = (pts[...,0]>=WORKSPACE_BOUNDS_MIN_RELAXED[0]) & (pts[...,0]<=WORKSPACE_BOUNDS_MAX_RELAXED[0])
                yin = (pts[...,1]>=WORKSPACE_BOUNDS_MIN_RELAXED[1]) & (pts[...,1]<=WORKSPACE_BOUNDS_MAX_RELAXED[1])
                zin = (pts[...,2]>=WORKSPACE_BOUNDS_MIN_RELAXED[2]) & (pts[...,2]<=WORKSPACE_BOUNDS_MAX_RELAXED[2])
                keep_b = (xin & yin & zin).all(axis=0)
                for suf in ['_scene_flows','_scene_colors','_scene_visibility','_scene_depth_valid_mask','_scene_normals']:
                    processed[clip_key][f'{cp}{suf}'] = processed[clip_key][f'{cp}{suf}'][:, keep_b]

        # Outlier removal
        if outlier_scope != 'none':
            for clip_key in list(processed.keys()):
                if ':' not in clip_key: continue
                cps = [k.replace('_scene_flows','') for k in processed[clip_key] if k.endswith('_scene_flows')]
                if not cps: continue
                if outlier_scope == 'global':
                    trajs = [processed[clip_key][f'{cp}_scene_flows'] for cp in cps]
                    viss = [processed[clip_key][f'{cp}_scene_visibility'] for cp in cps]
                    dvms = [processed[clip_key][f'{cp}_scene_depth_valid_mask'] for cp in cps]
                    counts = [t.shape[1] for t in trajs]
                    _, _, _, out_idx = remove_outlier_flows(
                        np.concatenate(trajs,axis=1), np.concatenate(viss,axis=1), np.concatenate(dvms,axis=1),
                        eps_values=list(outlier_eps_values), min_points=outlier_min_points,
                        min_outlier_frames_ratio=outlier_frames_ratio,
                    )
                    if out_idx is not None and len(out_idx)>0:
                        keep_g = np.ones(sum(counts), dtype=bool); keep_g[out_idx] = False
                        s=0
                        for cp, cnt in zip(cps, counts):
                            m = keep_g[s:s+cnt]
                            for suf in ['_scene_flows','_scene_colors','_scene_visibility','_scene_depth_valid_mask','_scene_normals']:
                                processed[clip_key][f'{cp}{suf}'] = processed[clip_key][f'{cp}{suf}'][:, m]
                            s += cnt
                else:  # per_camera
                    for cp in cps:
                        _, _, _, out_idx = remove_outlier_flows(
                            processed[clip_key][f'{cp}_scene_flows'],
                            processed[clip_key][f'{cp}_scene_visibility'],
                            processed[clip_key][f'{cp}_scene_depth_valid_mask'],
                            eps_values=list(outlier_eps_values), min_points=outlier_min_points,
                            min_outlier_frames_ratio=outlier_frames_ratio,
                        )
                        if out_idx is not None and len(out_idx)>0:
                            keep_p = np.ones(processed[clip_key][f'{cp}_scene_flows'].shape[1], dtype=bool)
                            keep_p[out_idx] = False
                            for suf in ['_scene_flows','_scene_colors','_scene_visibility','_scene_depth_valid_mask','_scene_normals']:
                                processed[clip_key][f'{cp}{suf}'] = processed[clip_key][f'{cp}{suf}'][:, keep_p]

        # Write
        self.scene_path = scene_path
        self.uuid = uuid
        flows_dir_name = 'flows-fs-optimize' if extrinsics_source == 'optimized' else f'flows-fs-{extrinsics_source}'
        flows_dir = os.path.join(output_dir, flows_dir_name)
        flows_h5 = os.path.join(flows_dir, f'{uuid}_flows.h5')
        os.makedirs(os.path.dirname(flows_h5), exist_ok=True)
        self._write_h5(
            processed, flows_h5,
            ee_pos_threshold=float(cache['ee_pos_threshold']),
            ee_rot_threshold=float(cache['ee_rot_threshold']),
            gripper_closed_ee_pos_threshold=float(cache['gripper_closed_ee_pos_threshold']),
            gripper_closed_ee_rot_threshold=float(cache['gripper_closed_ee_rot_threshold']),
        )
        print(f"[{get_time_str()}] Converted {uuid} → {flows_h5}")
        return flows_h5

print("✅ Flow2DTo3DConverter defined")

In [ ]:
# ============================================================
# Stage C — Run 2D→3D conversion
# ============================================================

# Read scene paths
with open(DROID_FLOW_INPUT, 'r') as f:
    convert_scene_paths = [line.strip() for line in f if line.strip()]

enforce_gcs_cache_policy(
    convert_scene_paths,
    stage_name='convert_2d_flows_to_3d',
    require_cache=True,
    allow_streaming=ALLOW_GCS_STREAMING,
)

# Resolve 2D flow cache files and validate prerequisites
cache_dir = os.path.join(DROID_ROOT, '2d_flows')
pairs = []
errors = []
for sp in convert_scene_paths:
    try:
        uuid = get_uuid(sp)
    except Exception as exc:
        errors.append(f"{sp}: failed uuid ({exc})"); continue
    cache_path = os.path.join(cache_dir, f"{uuid}_2d_flows.h5")
    depth_path = os.path.join(DROID_ROOT, 'depth', f"{uuid}_depth.h5")
    cameras_path = os.path.join(DROID_ROOT, 'cameras', f"{uuid}_cameras.json")
    missing = [p for p in [cache_path, depth_path, cameras_path] if not os.path.exists(p)]
    if missing:
        errors.append(f"{sp} ({uuid}): missing {', '.join(missing)}"); continue
    pairs.append((sp, cache_path))

if errors:
    print(f"⚠️ {len(errors)} scenes skipped due to missing artifacts:")
    for e in errors[:10]: print(f"  {e}")
    if len(errors) > 10: print(f"  ... and {len(errors)-10} more")

worker_pairs = pairs[RANK::WORLD_SIZE]
print(f"\nWorker {RANK}/{WORLD_SIZE} converting {len(worker_pairs)} scenes to 3D flows")

# Run conversion
conv = Flow2DTo3DConverter(urdf_path=URDF_PATH, robot_mask_seed=ROBOT_MASK_SEED)

converted = []
pbar = tqdm(worker_pairs, desc=f"worker {RANK} 3D convert", unit="scene", dynamic_ncols=True)
for scene_path, cache_path in pbar:
    tqdm.write(f"[{get_time_str()}] Converting: {os.path.basename(cache_path)}")
    res = conv.convert_scene(
        cache_path, DROID_ROOT,
        extrinsics_source=EXTRINSICS_SOURCE,
        outlier_eps_values=tuple(OUTLIER_EPS_VALUES),
        outlier_min_points=OUTLIER_MIN_POINTS,
        outlier_frames_ratio=OUTLIER_FRAMES_RATIO,
        outlier_scope=OUTLIER_SCOPE,
        scale_intrinsics_to_depth=SCALE_INTRINSICS_TO_DEPTH,
        seed_mask_mode=SEED_MASK_MODE,
    )
    if res: converted.append(res)

print(f"\n✅ Converted {len(converted)} scenes to 3D flows!")
if converted:
    print(f"Output directory: {os.path.dirname(converted[0])}")
    print(f"Example file: {converted[0]}")

---
## Summary

This notebook executed the full **filter + 2D/3D flow** pipeline:

| Stage | Description | Output |
|-------|------------|--------|
| **A** | Filter scenes by extrinsics quality | `droid_paths_final_loss_lt_0.10.txt` |
| **B** | Compute 2D flows (CoTracker) | `2d_flows/<uuid>_2d_flows.h5` |
| **C** | Convert 2D flows to 3D | `flows-fs-optimize/<uuid>_flows.h5` |

The final `<uuid>_flows.h5` files are ready for consumption by the PointWorld training pipeline.